<a href="https://colab.research.google.com/github/Razan445/2026-GP1-Group10-Mahd/blob/main/ai-audio-model/3_model_inference/Mahd_MLP_CubicSVM_kNN_Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mahd — Infant Cry Data Preparation

We prepare audio data for two tasks: **Cry Detection** (Cry / Non-Cry) and **Cry Cause Classification** (Hunger, Pain, Discomfort, Tired, Burping). We combine six datasets, exclude exact duplicates, check audio quality, assign data splits, and extract features from windows of up to 1 second. This notebook prepares the inputs; model training comes next.

| Section | Main output |
|---|---|
| 1. Setup | Libraries and dataset paths |
| 2. Datasets | Audio tables with a shared structure |
| 3. Merge and Duplicates | `clean_audio_df` (and `dup_status` for every file) |
| 4. Label Mapping | `mapped_df` |
| 5. Audio Preprocessing (Steps 1–5) | `load_audio`, `prepare_audio` |
| 6. Data Splitting | `split_df` |
| 7. Windows and File Selection | `work_df` and windowing functions |
| 8. Feature Extraction | `features_df` |
| Appendix A | Optional cells: audio inventory, durations, and the evidence for Steps 1–4 |

**Decisions applied in this notebook**

- Exact duplicates (same MD5) are excluded, keeping one copy (switch: `EXCLUDE_DUPLICATES`).
- No file is excluded for quality: recordings were checked by listening.
- Cry recordings lose only their leading and trailing silence (−70 dB); no cry window is deleted afterwards.
- Every file is only resampled to 16 kHz, and the spectral features are computed up to 4 kHz (Step 2). Peak normalisation targets 1.0 (Step 3).
- - 1-second windows with 50% overlap (hop 0.5 s); a file shorter than 1 s gives one shorter window of real audio, with no padding.
- Noisy recordings are kept, because the device operates in homes.

The notebook saves duplicate reports, split assignments, and feature chunks under `MyDrive/Mahd`.

# 1. Setup

We install and import the required libraries, mount Google Drive, and define the dataset paths. This gives later cells a consistent way to access audio and save results. ICSD and Donate-a-Cry are downloaded separately, so their paths are defined in their own sections.

In [ ]:
!pip install -q datasets huggingface_hub soundfile librosa pandas matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import re
import glob
import zlib
import hashlib
import zipfile
import warnings

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from pathlib import Path

In [ ]:
MAHD_ROOT = "/content/drive/MyDrive/Mahd"

OSF_ROOT = os.path.join(MAHD_ROOT, "OSF")
NONSPEECH_ROOT = os.path.join(MAHD_ROOT, "NonSpeech7k")

DBL_FOLDER = next(
    x for x in os.listdir(MAHD_ROOT)
    if x.startswith("InfantCry-DBL")
)
DBL_ROOT = os.path.join(MAHD_ROOT, DBL_FOLDER)

SCHEMA = [
    "audio_path",
    "dataset",
    "subset",
    "original_label",
    "source_id",
    "source_id_type",
    "orig_split"
]

# 2. Datasets

We load each dataset into the same table structure (`SCHEMA`). Keeping the original labels and source identifiers lets us merge the tables now, then map labels and assign splits without losing their origin.

| Column | Meaning |
|---|---|
| `audio_path` | Full path to the audio file |
| `dataset` | Dataset name |
| `subset` | Collection or tier within the dataset |
| `original_label` | Original label, before mapping |
| `source_id` | Available infant, recording, or app identifier |
| `source_id_type` | What the identifier represents |
| `orig_split` | Official split, when available |

## 2.1 ICSD

ICSD is an **infant cry and snoring detection** dataset. Its Strong data contain labelled infant-cry and snoring recordings.

The Strong data include real and synthetic recordings, so these sources are kept identifiable during preparation.


### Download ICSD

Hugging Face authentication is used to download and extract the ICSD files and metadata needed for the project.


In [ ]:
from huggingface_hub import login
login()

In [ ]:
from huggingface_hub import hf_hub_download
import os
import zipfile
import pandas as pd

icsd_dir = "/content/datasets/ICSD"
os.makedirs(icsd_dir, exist_ok=True)

zip_path = hf_hub_download(
    repo_id="QingyuLiu1/ICSD",
    filename="Dataset.zip",
    repo_type="dataset"
)

extract_dir = os.path.join(icsd_dir, "Dataset")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

Dataset.zip: reconstructing file:   0%|          |  0.00B / 6.67GB            

Dataset.zip: downloading bytes:           |  0.00B            

### Load Strong Metadata

Only the **Real Strong** and **Synthetic Strong** metadata are used. The Weak subset is excluded because it contains recordings from datasets that are already included separately, such as Donate-a-Cry. This helps reduce source overlap.


In [ ]:
metadata_paths = {
    "Real Train": "metadata/train/real_strong_train.tsv",
    "Synthetic Train": "metadata/train/synth_strong_train.tsv",
    "Validation": "metadata/validation/strong_val.tsv",
    "Real Test": "metadata/test/real_strong_test.tsv",
    "Synthetic Test": "metadata/test/synth_strong_test.tsv"
}

metadata = {
    name: pd.read_csv(os.path.join(extract_dir, path), sep="\t")
    for name, path in metadata_paths.items()
}

### Separate Validation Recordings

Real and synthetic validation recordings are kept identifiable using their filename prefixes.


In [ ]:
val_df = metadata["Validation"]

real_val = val_df[
    val_df["filename"].str.startswith("strong_")
].copy()

synth_val = val_df[
    val_df["filename"].str.startswith("synth_")
].copy()

### ICSD Class Distribution

This cell counts unique recordings by event label to check the class distribution before further processing.


In [ ]:
subsets = {
    "Real Train": metadata["Real Train"],
    "Real Validation": real_val,
    "Real Test": metadata["Real Test"],
    "Synthetic Train": metadata["Synthetic Train"],
    "Synthetic Validation": synth_val,
    "Synthetic Test": metadata["Synthetic Test"]
}

rows = []

for name, df in subsets.items():
    unique_files = df[["filename", "event_label"]].drop_duplicates()
    counts = unique_files["event_label"].value_counts()

    rows.append({
        "Subset": name,
        "Infant Cry": counts.get("Infantcry", 0),
        "Snoring": counts.get("Snoring", 0),
        "Total": len(unique_files)
    })

icsd_distribution = pd.DataFrame(rows)

icsd_distribution["Infant Cry %"] = (
    icsd_distribution["Infant Cry"] /
    icsd_distribution["Total"] * 100
).round(2)

icsd_distribution["Snoring %"] = (
    icsd_distribution["Snoring"] /
    icsd_distribution["Total"] * 100
).round(2)

display(icsd_distribution)

,Subset,Infant Cry,Snoring,Total,Infant Cry %,Snoring %
0,Real Train,338,305,643,52.57,47.43
1,Real Validation,43,39,82,52.44,47.56
2,Real Test,43,39,82,52.44,47.56
3,Synthetic Train,4000,4000,8000,50.00,50.00
4,Synthetic Validation,500,500,1000,50.00,50.00
5,Synthetic Test,500,500,1000,50.00,50.00


### ICSD Materials

The Materials archive contains source sounds used to create synthetic recordings.

- **Foreground:** infant cry and snoring
- **Background:** speech, music, cooking, TV, and other sounds

The background sounds are used as candidate **Non-Cry** examples.


In [ ]:
from huggingface_hub import hf_hub_download
import os
import zipfile

materials_zip = hf_hub_download(
    repo_id="QingyuLiu1/ICSD",
    filename="Materials.zip",
    repo_type="dataset"
)

materials_dir = "/content/datasets/ICSD/Materials"

os.makedirs(materials_dir, exist_ok=True)

with zipfile.ZipFile(materials_zip, "r") as zip_ref:
    zip_ref.extractall(materials_dir)

print("Extracted to:", materials_dir)

Materials.zip: reconstructing file:   0%|          |  0.00B / 20.0GB            

Materials.zip: downloading bytes:           |  0.00B            

Extracted to: /content/datasets/ICSD/Materials


### Build ICSD Audio Table

Metadata filenames are matched to their audio files. Only unambiguous matches are accepted, so a recording is not assigned to the wrong file.


In [ ]:
# Index every audio file in the extracted ICSD Dataset folder by its filename
icsd_index = {}

for root, dirs, files in os.walk(extract_dir):
    for file in files:
        if file.lower().endswith((".wav", ".flac", ".mp3", ".ogg")):
            icsd_index.setdefault(file, []).append(os.path.join(root, file))


icsd_subsets = [
    ("Real Strong", metadata["Real Train"], "train"),
    ("Synthetic Strong", metadata["Synthetic Train"], "train"),
    ("Real Strong", real_val, "val"),
    ("Synthetic Strong", synth_val, "val"),
    ("Real Strong", metadata["Real Test"], "test"),
    ("Synthetic Strong", metadata["Synthetic Test"], "test"),
]

records = []
unresolved = []

for subset_name, df, split in icsd_subsets:
    unique_files = df[["filename", "event_label"]].drop_duplicates()

    for _, row in unique_files.iterrows():
        matches = icsd_index.get(row["filename"], [])

        if len(matches) != 1:
            unresolved.append((subset_name, split, row["filename"], len(matches)))
            continue

        records.append({
            "audio_path": matches[0],
            "dataset": "ICSD",
            "subset": subset_name,
            "original_label": row["event_label"],
            "source_id": None,
            "source_id_type": "none",
            "orig_split": split
        })

icsd_df = pd.DataFrame(records, columns=SCHEMA)

print("ICSD Strong files:", len(icsd_df))
print("Unresolved filenames:", len(unresolved))

ICSD Strong files: 10807
Unresolved filenames: 0


### Build the Background Audio Table

Background files are collected with their original source labels and kept under `Materials Background`.


In [ ]:
# Real background recordings (speech, music, cooking, ...) used to build the
# Synthetic Strong recordings. Kept as a separate subset so they can be
# included or excluded explicitly later.
background_dir = os.path.join(materials_dir, "background")

records = []

for class_name in sorted(os.listdir(background_dir)):
    class_dir = os.path.join(background_dir, class_name)

    if not os.path.isdir(class_dir):
        continue

    for file in sorted(os.listdir(class_dir)):
        if file.lower().endswith(".wav"):
            records.append({
                "audio_path": os.path.join(class_dir, file),
                "dataset": "ICSD",
                "subset": "Materials Background",
                "original_label": class_name,
                "source_id": None,
                "source_id_type": "none",
                "orig_split": None
            })

icsd_bg_df = pd.DataFrame(records, columns=SCHEMA)

print("Materials Background files:", len(icsd_bg_df))
display(icsd_bg_df["original_label"].value_counts())

Materials Background files: 83815


,count
original_label,
absence,18860
watching_tv,18648
working,18644
speech,5400
cooking,5124
social_activity,4944
music,3600
eating,2308
other,2060


## 2.2 Baby Chillanto

Baby Chillanto contains medically recorded infant cries. The selected data provide **Hunger** and **Pain** recordings.

Full recordings and 1-second segments can overlap, so only the full recordings are used here. Recordings sharing the same leading number are grouped together.

### Load Baby Chillanto





In [ ]:
DATASET_PATH = Path("/content/drive/MyDrive/Mahd/Baby Chillanto")

print("Dataset path:", DATASET_PATH)
print("Exists:", DATASET_PATH.exists())

Dataset path: /content/drive/MyDrive/Mahd/Baby Chillanto
Exists: True


### Inspect the Dataset

In [ ]:
folders = [f for f in DATASET_PATH.iterdir() if f.is_dir()]

print("Number of folders:", len(folders))

for folder in folders:
    print(folder.name)

Number of folders: 4
Full_hunger
Full_pain
1s_hunger
1s_pain


In [ ]:
audio_extensions = {".wav", ".mp3", ".flac", ".ogg"}

audio_files = []

for file in DATASET_PATH.rglob("*"):
    if file.is_file() and file.suffix.lower() in audio_extensions:
        audio_files.append(file)

print("Total audio files found:", len(audio_files))

Total audio files found: 601


In [ ]:
for folder in sorted(DATASET_PATH.iterdir()):
    if folder.is_dir():
        files = [
            f for f in folder.rglob("*")
            if f.is_file() and f.suffix.lower() in audio_extensions
        ]

        print(f"{folder.name}: {len(files)} files")

1s_hunger: 350 files
1s_pain: 192 files
Full_hunger: 34 files
Full_pain: 25 files


### Prepare Metadata

In [ ]:
#Create Metadata Records
records = []

for file in DATASET_PATH.rglob("*"):
    if file.is_file() and file.suffix.lower() in audio_extensions:

        path_lower = str(file).lower()

        # Label
        if "hunger" in path_lower:
            label = "hunger"
        elif "pain" in path_lower:
            label = "pain"
        else:
            label = "unknown"

        # Recording type
        if "1s_" in path_lower:
            recording_type = "segmented_1s"
        elif "full_" in path_lower:
            recording_type = "full"
        else:
            recording_type = "unknown"

        records.append({
            "filename": file.name,
            "filepath": str(file),
            "label": label,
            "recording_type": recording_type
        })

chillanto_metadata = pd.DataFrame(records)

print("Total audio files:", len(chillanto_metadata))

Total audio files: 601


In [ ]:
chillanto_metadata["label"].value_counts()

,count
label,
hunger,384
pain,217


In [ ]:
chillanto_metadata["recording_type"].value_counts()

,count
recording_type,
segmented_1s,542
full,59


In [ ]:
pd.crosstab(
    chillanto_metadata["label"],
    chillanto_metadata["recording_type"]
)

recording_type,full,segmented_1s
label,,
hunger,34,350
pain,25,192


### Audio Properties

In [ ]:
import soundfile as sf


In [ ]:
audio_info = []

for _, row in chillanto_metadata.iterrows():
    try:
        info = sf.info(row["filepath"])

        audio_info.append({
            "filename": row["filename"],
            "label": row["label"],
            "recording_type": row["recording_type"],
            "duration_sec": info.duration,
            "sample_rate": info.samplerate,
            "channels": info.channels,
            "subtype": info.subtype
        })

    except Exception as e:
        audio_info.append({
            "filename": row["filename"],
            "label": row["label"],
            "recording_type": row["recording_type"],
            "duration_sec": np.nan,
            "sample_rate": np.nan,
            "channels": np.nan,
            "subtype": "ERROR"
        })

audio_metadata = pd.DataFrame(audio_info)

print("Total files:", len(audio_metadata))

Total files: 601


In [ ]:
audio_metadata.groupby("recording_type")["duration_sec"].describe()

,count,mean,std,min,25%,50%,75%,max
recording_type,,,,,,,,
full,59.0,9.708958,6.089753,1.0,5.87546,9.0055,12.533346,39.948481
segmented_1s,542.0,1.000000,0.000000,1.0,1.00000,1.0000,1.000000,1.000000


In [ ]:
audio_metadata["sample_rate"].value_counts()

,count
sample_rate,
8000,302
11025,188
22050,111


In [ ]:
audio_metadata["channels"].value_counts()

,count
channels,
1,601


### Full vs 1-Second Recordings

According to the dataset providers, Baby Chillanto contains two categories of samples:

1. Complete recordings of individual infant cries.
2. One-second segments derived from the same complete recordings.

Therefore, the one-second recordings are not considered completely independent from the corresponding full recordings. This relationship must be considered when creating training, validation, and test splits to avoid data leakage.

In [ ]:
full_metadata = audio_metadata[
    audio_metadata["recording_type"] == "full"
].copy()

full_metadata[
    ["filename", "label", "duration_sec", "sample_rate", "channels"]
].sort_values("duration_sec")

,filename,label,duration_sec,sample_rate,channels
36,10a.wav,pain,1.000000,8000,1
34,9b.wav,pain,1.944000,8000,1
40,17.wav,pain,3.014104,22050,1
37,14.wav,pain,3.514966,22050,1
41,19.wav,pain,3.582494,22050,1
23,5.wav,hunger,3.680454,22050,1
43,22.wav,pain,3.854467,22050,1
55,6.wav,pain,4.300408,22050,1
58,9a.wav,pain,4.578875,8000,1
47,45.wav,pain,4.692750,8000,1


In [ ]:
full_metadata.groupby("label")["duration_sec"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
hunger,34.0,10.737639,4.664402,3.680454,7.415431,9.783300,12.841156,26.377868
pain,25.0,8.309953,7.494603,1.000000,4.300408,6.360862,9.582449,39.948481


In [ ]:
full_metadata["sample_rate"].value_counts()

,count
sample_rate,
8000,31
22050,17
11025,11


### Build the Baby Chillanto Audio Table


In [ ]:
CHILLANTO_ROOT = os.path.join(MAHD_ROOT, "Baby Chillanto")

CHILLANTO_SEGMENT_FOLDERS = {
    "1s_hunger": ("hunger", "001"),
    "1s_pain": ("pain", "002"),
}

SEGMENT_NAME = re.compile(r"^(\d{4})(\d{3})(\d{3})\.wav$", re.IGNORECASE)

records = []

for folder, (label, class_code) in CHILLANTO_SEGMENT_FOLDERS.items():
    for root, dirs, files in os.walk(os.path.join(CHILLANTO_ROOT, folder)):   # segments are in a "1 seg" subfolder
        for file in sorted(files):
            match = SEGMENT_NAME.match(file)
            if match is None:
                continue

            recording, segment, code = match.groups()

            records.append({
                "audio_path": os.path.join(root, file),
                "dataset": "Baby Chillanto",
                "subset": "1-second segments",
                "original_label": label,
                "source_id": str(int(recording)),      # "0004" -> "4"
                "source_id_type": "recording_number",
                "orig_split": None,
            })

chillanto_df = pd.DataFrame(records, columns=SCHEMA)

print("Total segments:", len(chillanto_df))
print(chillanto_df["original_label"].value_counts())

Total segments: 542
original_label
hunger    350
pain      192
Name: count, dtype: int64


In [ ]:
chillanto_distribution = (
    chillanto_df.groupby("original_label")
    .agg(Segments=("audio_path", "size"), Recordings=("source_id", "nunique"))
    .reset_index()
    .rename(columns={"original_label": "Class"})
)
chillanto_distribution["Percentage"] = (
    chillanto_distribution["Segments"] / chillanto_distribution["Segments"].sum() * 100
).round(2)

display(chillanto_distribution)

,Class,Segments,Recordings,Percentage
0,hunger,350,33,64.58
1,pain,192,23,35.42


## 2.3 Donate-a-Cry

Donate-a-Cry contains contributed infant cries labelled with the **suspected reason for crying**:

- **Hungry:** suspected hunger.
- **Discomfort:** suspected physical discomfort.
- **Tired:** suspected tiredness.
- **Belly Pain:** suspected abdominal pain.
- **Burping:** suspected need to burp.

These are parent-reported labels, not clinically confirmed diagnoses. The selected cleaned-and-updated version contains 457 recordings. Its filename identifier represents an app installation, not a verified infant identity.

### Download Donate-a-Cry


In [ ]:
!git clone -q https://github.com/gveres/donateacry-corpus.git /content/datasets/donateacry-corpus

donate_dir = "/content/datasets/donateacry-corpus"

### Select the Corpus Version

Only `donateacry_corpus_cleaned_and_updated_data` is scanned. Restricting the search to this directory avoids counting other repository versions together.

In [ ]:
import os
import pandas as pd

donate_clean_dir = os.path.join(
    donate_dir,
    "donateacry_corpus_cleaned_and_updated_data"
)

donate_files = []

for root, dirs, files in os.walk(donate_clean_dir):
    for file in files:
        if file.lower().endswith((".wav", ".caf", ".3gp", ".mp3", ".m4a", ".ogg")):
            donate_files.append(os.path.join(root, file))

print("Total audio files:", len(donate_files))

Total audio files: 457


### Build the Donate-a-Cry Audio Table

The folder names provide the original cry labels. A group ID is extracted from the filename to keep related recordings together during data splitting and help prevent data leakage.

In [ ]:
records = []

for path in donate_files:
    filename = os.path.basename(path)
    label = os.path.basename(os.path.dirname(path))

    records.append({
        "audio_path": path,
        "dataset": "Donate-a-Cry",
        "subset": "cleaned_and_updated_data",
        "original_label": label,
        "source_id": filename[:36].lower(),
        "source_id_type": "app_installation_uuid",
        "orig_split": None
    })

donate_df = pd.DataFrame(records, columns=SCHEMA)

print("Discovered classes:")
print(sorted(donate_df["original_label"].unique()))
print(donate_df.columns.tolist())

Discovered classes:
['belly_pain', 'burping', 'discomfort', 'hungry', 'tired']
['audio_path', 'dataset', 'subset', 'original_label', 'source_id', 'source_id_type', 'orig_split']


### Donate-a-Cry Class Distribution

Recordings are counted by original label, with percentages to show imbalance. These counts describe the selected corpus before duplicate removal.

In [ ]:
donate_distribution = (
    donate_df["original_label"]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)

donate_distribution["Percentage"] = (
    donate_distribution["Count"]
    / donate_distribution["Count"].sum()
    * 100
).round(2)

display(donate_distribution)

print("Total:", len(donate_df))

,Class,Count,Percentage
0,hungry,382,83.59
1,discomfort,27,5.91
2,tired,24,5.25
3,belly_pain,16,3.50
4,burping,8,1.75


Total: 457


## 2.4 OSF

This dataset accompanies a controlled study of infant cries. Its natural recordings cover three documented conditions:

- **Discomfort:** bathing, dressing, or undressing.
- **Acute Pain:** the moment of vaccine injection.
- **Residual Pain:** approximately six seconds after injection.

The study also includes synthetic and playback materials. Only natural cries are selected because they are actual infant recordings collected under the relevant conditions. The metadata provide `baby_ID` to identify recordings from the same infant.

### Load Natural Cry Records

The code reads `cries_description.csv` and links each recording to `baby_ID`. One known filename mismatch is corrected so the file can be matched to its metadata.

In [ ]:
OSF_NATURAL_DIR = os.path.join(OSF_ROOT, "Natural cries")

osf_description = pd.read_csv(
    os.path.join(OSF_NATURAL_DIR, "cries_description.csv"),
    sep=";"
)

# One baby_ID per recording
osf_baby_ids = (
    osf_description
    .drop_duplicates("file")
    .set_index("file")["baby_ID"]
)

# Recording name on disk -> name in cries_description.csv
OSF_NAME_FIXES = {
    "norm_VIC_B2_F_2014-04-22-BB24_01min47_crac&water":
    "norm_VIC_B2_F_2014-04-22-BB24_01min47_crac_water"
}

records = []

for folder in os.listdir(OSF_NATURAL_DIR):
    folder_path = os.path.join(OSF_NATURAL_DIR, folder)

    if folder.startswith("Data -") and os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            if file.lower().endswith(".wav"):
                name = os.path.splitext(file)[0]
                name = OSF_NAME_FIXES.get(name, name)

                records.append({
                    "audio_path": os.path.join(folder_path, file),
                    "dataset": "OSF",
                    "subset": "Natural Cries",
                    "original_label": folder,
                    "source_id": osf_baby_ids.get(name),
                    "source_id_type": "infant_id",
                    "orig_split": None
                })

osf_df = pd.DataFrame(records, columns=SCHEMA)

print("Discovered classes:")
print(sorted(osf_df["original_label"].unique()))
print("\nTotal audio files:", len(osf_df))
print("Infants:", osf_df["source_id"].nunique())

Discovered classes:
['Data - acute pain cries', 'Data - discomfort cries', 'Data - residual pain cries']

Total audio files: 304
Infants: 24


### Standardize OSF Labels

Organisational text such as `Data -` and `cries` is removed from folder labels. This leaves consistent names for the three study conditions without changing their meaning.

In [ ]:
osf_df["original_label"] = (
    osf_df["original_label"]
    .str.replace("Data - ", "", regex=False)
    .str.replace(" cries", "", regex=False)
    .str.title()
)

print(sorted(osf_df["original_label"].unique()))

['Acute Pain', 'Discomfort', 'Residual Pain']


### OSF Class Distribution

In [ ]:
osf_distribution = (
    osf_df["original_label"]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)

osf_distribution["Percentage"] = (
    osf_distribution["Count"]
    / osf_distribution["Count"].sum()
    * 100
).round(2)

display(osf_distribution)

print("Total:", len(osf_df))

,Class,Count,Percentage
0,Discomfort,210,69.08
1,Acute Pain,48,15.79
2,Residual Pain,46,15.13


Total: 304


## 2.5 InfantCry-DBL

InfantCry-DBL contains cries labelled using **Dunstan Baby Language (DBL)** categories, which associate sound patterns with possible infant needs:

| Category | Associated need or condition |
|---|---|
| `neh` | Hunger |
| `owh` | Tiredness |
| `heh` | Physical discomfort |
| `eairh` | Lower-wind / abdominal discomfort |
| `eh` | Need to burp |

**Dunstan-Core (Tier 1)** contains 337 clips from instructional videos. **InfantCry-1214 (Tier 2)** contains 1,214 clips from CryCeleb2023, online sources, and volunteer homes. Labels are annotations of sound categories, not confirmed infant needs. Tier 1 has no infant IDs; Tier 2 filename IDs are not clearly defined in the README.

### Build the DBL Audio Table

The table is built from `metadata.csv`, using `class_folder` as the authoritative label. The Tier 2 folder spelling is corrected to match disk, and filename IDs are preserved as text. Tier 1 records have no source ID.

In [ ]:
DBL_DATASET_DIR = os.path.join(DBL_ROOT, DBL_FOLDER, "Datasets")

dbl_metadata = pd.read_csv(
    os.path.join(DBL_DATASET_DIR, "metadata.csv"),
    dtype={"filename": str}
)

records = []

for _, row in dbl_metadata.iterrows():
    relative_path = row["relative_path"].replace(
        "1214 COLLCTED DATA",
        "1214 Collected Data"
    )

    if row["tier"] == 2:
        source_id = row["filename"].split("_")[0]   # kept as text, e.g. "000124"
        source_id_type = "filename_id_undocumented"
    else:
        source_id = None
        source_id_type = "none"

    records.append({
        "audio_path": os.path.join(DBL_DATASET_DIR, relative_path),
        "dataset": "InfantCry-DBL",
        "subset": row["tier_name"],
        "original_label": row["class_folder"],
        "source_id": source_id,
        "source_id_type": source_id_type,
        "orig_split": None
    })

dbl_df = pd.DataFrame(records, columns=SCHEMA)

print("Total audio files:", len(dbl_df))
print("\nDiscovered classes:")
print(sorted(dbl_df["original_label"].unique()))

Total audio files: 1551

Discovered classes:
['eairh', 'eh', 'heh', 'neh', 'owh']


### DBL Class Distribution



In [ ]:
dbl_distribution = (
    dbl_df
    .groupby(["subset", "original_label"])
    .size()
    .reset_index(name="Count")
    .rename(columns={"subset": "Subset", "original_label": "Class"})
)

dbl_distribution["Percentage"] = (
    dbl_distribution["Count"]
    / dbl_distribution.groupby("Subset")["Count"].transform("sum")
    * 100
).round(2)

display(dbl_distribution)

print("Total:", dbl_distribution["Count"].sum())

,Subset,Class,Count,Percentage
0,Dunstan-Core,eairh,60,17.80
1,Dunstan-Core,eh,72,21.36
2,Dunstan-Core,heh,57,16.91
3,Dunstan-Core,neh,72,21.36
4,Dunstan-Core,owh,76,22.55
5,InfantCry-1214,eairh,110,9.06
6,InfantCry-1214,eh,208,17.13
7,InfantCry-1214,heh,111,9.14
8,InfantCry-1214,neh,145,11.94
9,InfantCry-1214,owh,640,52.72


Total: 1551


## 2.6 Nonspeech7k

Nonspeech7k contains seven human non-speech sound classes: **breath, cough, crying, laugh, screaming, sneeze, and yawn**, with official train and test metadata.

The dataset is not infant-specific. Its `crying` class is therefore not assumed to represent infant crying; the other classes provide examples of sounds that an infant-cry detector should distinguish from crying.

### Load Metadata and Fix the Typo

Official train and test metadata are loaded with `File ID` preserved as text. The test label `yawm` is corrected to `yawn` so one class is not counted under two spellings.

In [ ]:
train_df = pd.read_csv(
    os.path.join(NONSPEECH_ROOT, "metadata of train set .csv"),
    dtype={"File ID": str}
)

test_df = pd.read_csv(
    os.path.join(NONSPEECH_ROOT, "metadata of test set.csv"),
    dtype={"File_ID": str}
)

train_df["Classname"] = (
    train_df["Classname"]
    .str.strip()
    .str.lower()
    .replace({"yawm": "yawn"})
)

test_df["Classname"] = (
    test_df["Classname"]
    .str.strip()
    .str.lower()
    .replace({"yawm": "yawn"})
)

### Nonspeech7k Class Distribution

In [ ]:
train_counts = train_df["Classname"].value_counts()
test_counts = test_df["Classname"].value_counts()

classes = sorted(set(train_counts.index) | set(test_counts.index))

rows = []

for class_name in classes:
    train_count = train_counts.get(class_name, 0)
    test_count = test_counts.get(class_name, 0)

    rows.append({
        "Class": class_name,
        "Train": train_count,
        "Test": test_count,
        "Total": train_count + test_count
    })

nonspeech_distribution = pd.DataFrame(rows)

nonspeech_distribution["Percentage"] = (
    nonspeech_distribution["Total"]
    / nonspeech_distribution["Total"].sum()
    * 100
).round(2)

nonspeech_distribution = (
    nonspeech_distribution
    .sort_values("Total", ascending=False)
    .reset_index(drop=True)
)

display(nonspeech_distribution)

print("Total:", nonspeech_distribution["Total"].sum())

,Class,Train,Test,Total,Percentage
0,crying,1791,205,1996,28.46
1,breath,1690,160,1850,26.38
2,laugh,1133,140,1273,18.15
3,cough,632,70,702,10.01
4,screaming,593,70,663,9.45
5,sneeze,236,30,266,3.79
6,yawn,214,50,264,3.76


Total: 7014


### Build the Nonspeech7k Audio Table



In [ ]:
# Build the Nonspeech7k audio table from the metadata.
# All seven classes are kept, including `crying`, so the distributions stay
# complete; `crying` is excluded later in the mapping section.
# The train and test CSVs name the ID column differently.

nonspeech_splits = [
    ("train", train_df, "File ID"),
    ("test", test_df, "File_ID"),
]

records = []

for split, df, id_column in nonspeech_splits:
    audio_dir = os.path.join(NONSPEECH_ROOT, split, split)

    for _, row in df.iterrows():
        records.append({
            "audio_path": os.path.join(audio_dir, row["Filename"]),
            "dataset": "Nonspeech7k",
            "subset": row["source"],
            "original_label": row["Classname"],
            "source_id": row[id_column],
            "source_id_type": "file_id_undocumented",
            "orig_split": split
        })

nonspeech_df = pd.DataFrame(records, columns=SCHEMA)

print("Total audio files:", len(nonspeech_df))
print(nonspeech_df["orig_split"].value_counts())

Total audio files: 7014
orig_split
train    6289
test      725
Name: count, dtype: int64


# 3. Merge and Duplicates

The audio tables of all datasets are combined into one table with the same
columns. At this point nothing is mapped, filtered, or split: every file keeps
its original label, so the combined table can be checked against the
per-dataset distributions above.

In [ ]:
all_audio_df = pd.concat(
    [
        icsd_df,
        icsd_bg_df,
        donate_df,
        osf_df,
        dbl_df,
        nonspeech_df,
        chillanto_df
    ],
    ignore_index=True
)

print("Total audio files:", len(all_audio_df))

display(
    all_audio_df
    .groupby(["dataset", "subset"])
    .size()
    .reset_index(name="Files")
)

Total audio files: 104490


,dataset,subset,Files
0,Baby Chillanto,1-second segments,542
1,Donate-a-Cry,cleaned_and_updated_data,457
2,ICSD,Materials Background,83815
3,ICSD,Real Strong,807
4,ICSD,Synthetic Strong,10000
5,InfantCry-DBL,Dunstan-Core,337
6,InfantCry-DBL,InfantCry-1214,1214
7,Nonspeech7k,https://aigei.com/,1739
8,Nonspeech7k,https://freesound.org/,3660
9,Nonspeech7k,https://youtube.com/,1615


### Find Duplicates

Each file receives an MD5 fingerprint. Matching hashes identify byte-identical files even when filenames differ; trimmed or re-encoded copies are not detected. Hashes are cached in `file_hashes.csv`, and the report groups duplicates by source, label, and split.

In [ ]:
import os
import hashlib
import pandas as pd
from tqdm.auto import tqdm

HASH_CACHE_PATH = os.path.join(MAHD_ROOT, "file_hashes.csv")


def file_md5(path):
    """MD5 of the file content, read in 1 MB chunks."""
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


# Load hashes computed in a previous session, if any
if os.path.exists(HASH_CACHE_PATH):
    hash_cache = pd.read_csv(HASH_CACHE_PATH)
    known_hashes = dict(zip(hash_cache["audio_path"], hash_cache["md5"]))
else:
    known_hashes = {}

to_compute = [p for p in all_audio_df["audio_path"] if p not in known_hashes]
print("Already hashed:", len(known_hashes), "| to compute:", len(to_compute))

for i, path in enumerate(tqdm(to_compute, desc="Hashing files")):
    known_hashes[path] = file_md5(path)

    # Save progress every 5,000 files in case the session disconnects
    if (i + 1) % 5000 == 0:
        pd.DataFrame(list(known_hashes.items()), columns=["audio_path", "md5"]) \
          .to_csv(HASH_CACHE_PATH, index=False)

pd.DataFrame(list(known_hashes.items()), columns=["audio_path", "md5"]) \
  .to_csv(HASH_CACHE_PATH, index=False)

all_audio_df["md5"] = all_audio_df["audio_path"].map(known_hashes)
print("Hashed files:", all_audio_df["md5"].notna().sum())

Already hashed: 104549 | to compute: 0


Hashing files: 0it [00:00, ?it/s]

Hashed files: 104490


In [ ]:
duplicated_rows = all_audio_df[all_audio_df.duplicated("md5", keep=False)].copy()
duplicated_rows["location"] = duplicated_rows["dataset"] + " / " + duplicated_rows["subset"]

# One row per duplicated recording
duplicate_groups = duplicated_rows.groupby("md5").agg(
    found_in=("location", lambda s: " + ".join(sorted(set(s)))),
    labels=("original_label", lambda s: " | ".join(sorted(set(s)))),
    splits=("orig_split", lambda s: " | ".join(sorted(set(s.dropna())))),
    copies=("audio_path", "size"),
)

# Summary: where, which labels, which splits
duplicate_report = (
    duplicate_groups
    .groupby(["found_in", "labels", "splits"])
    .agg(recordings=("copies", "size"), files=("copies", "sum"))
    .reset_index()
)

pd.set_option("display.max_colwidth", None)
print("Files with at least one identical copy:", len(duplicated_rows))
print("Distinct duplicated recordings:", len(duplicate_groups))
display(duplicate_report)

Files with at least one identical copy: 540
Distinct duplicated recordings: 269


,found_in,labels,splits,recordings,files
0,Donate-a-Cry / cleaned_and_updated_data + InfantCry-DBL / InfantCry-1214,eh | hungry,,13,26
1,ICSD / Materials Background,absence,,55,110
2,ICSD / Real Strong,Infantcry | Snoring,test | train,1,2
3,ICSD / Real Strong,Infantcry | Snoring,train,1,2
4,ICSD / Real Strong,Snoring,test | train,1,2
5,InfantCry-DBL / Dunstan-Core + InfantCry-DBL / InfantCry-1214,eairh,,19,38
6,InfantCry-DBL / Dunstan-Core + InfantCry-DBL / InfantCry-1214,eh,,45,92
7,InfantCry-DBL / Dunstan-Core + InfantCry-DBL / InfantCry-1214,eh | heh,,2,4
8,InfantCry-DBL / Dunstan-Core + InfantCry-DBL / InfantCry-1214,heh,,35,70
9,InfantCry-DBL / Dunstan-Core + InfantCry-DBL / InfantCry-1214,heh | neh,,1,2


### Remove Duplicates

The duplicate report above identifies exact overlaps within DBL, between DBL and Donate-a-Cry, and within ICSD. Duplicates are removed from the table using the first applicable rule:

1. **Donate-a-Cry / DBL overlap:** remove the DBL copy.
2. **Other conflicting labels:** remove all copies because the correct label is uncertain.
3. **Matching labels:** keep one copy, preferring the official test split, then DBL Tier 1.

Files remain on Drive. Removal reasons are saved in `removed_duplicates.csv`, and the retained table is `clean_audio_df`.

`all_audio_df["dup_status"]` records what happened to every file (`unique`, `kept_copy`, or `flagged: <reason>`). The switch `EXCLUDE_DUPLICATES = False` keeps every file instead (for example, if a supervisor asks for no exclusions).

In [ ]:
removed = []

for md5, group in duplicated_rows.groupby("md5"):
    datasets = set(group["dataset"])
    labels = set(group["original_label"])

    # Rule 1: Donate-a-Cry recording copied into InfantCry-DBL
    if "Donate-a-Cry" in datasets and "InfantCry-DBL" in datasets:
        for path in group.loc[group["dataset"] == "InfantCry-DBL", "audio_path"]:
            removed.append({"audio_path": path,
                            "reason": "InfantCry-DBL copy of a Donate-a-Cry recording"})
        continue

    # Rule 2: same recording, different labels
    if len(labels) > 1:
        for path in group["audio_path"]:
            removed.append({"audio_path": path,
                            "reason": "Identical recording with conflicting labels"})
        continue

    # Rule 3: same label -> keep one copy (test split first, then DBL Tier 1)
    group = group.assign(
        split_priority=[0 if s == "test" else 1 for s in group["orig_split"]],
        tier_priority=[0 if s == "Dunstan-Core" else 1 for s in group["subset"]],
    ).sort_values(["split_priority", "tier_priority", "audio_path"])

    for path in group["audio_path"].iloc[1:]:
        removed.append({"audio_path": path,
                        "reason": "Identical copy of a kept recording"})


removed_df = pd.DataFrame(removed).merge(
    all_audio_df[["audio_path", "dataset", "subset", "original_label", "orig_split", "md5"]],
    on="audio_path"
)
removed_df.to_csv(os.path.join(MAHD_ROOT, "removed_duplicates.csv"), index=False)

EXCLUDE_DUPLICATES = True      # False = keep every file

# Status of every file
all_audio_df["dup_status"] = "unique"
all_audio_df.loc[all_audio_df.duplicated("md5", keep=False), "dup_status"] = "kept_copy"

reason_by_path = removed_df.set_index("audio_path")["reason"]
is_dup = all_audio_df["audio_path"].isin(reason_by_path.index)
all_audio_df.loc[is_dup, "dup_status"] = (
    "flagged: " + all_audio_df.loc[is_dup, "audio_path"].map(reason_by_path)
)

if EXCLUDE_DUPLICATES:
    clean_audio_df = all_audio_df[~is_dup][SCHEMA].reset_index(drop=True)
else:
    clean_audio_df = all_audio_df[SCHEMA].reset_index(drop=True)

print("Before:", len(all_audio_df), "| flagged:", int(is_dup.sum()), "| after:", len(clean_audio_df))
display(
    removed_df
    .groupby(["reason", "dataset", "subset"])
    .size()
    .reset_index(name="Files flagged")
)

Before: 104490 | flagged: 276 | after: 104214


,reason,dataset,subset,Files flagged
0,Identical copy of a kept recording,ICSD,Materials Background,55
1,Identical copy of a kept recording,ICSD,Real Strong,1
2,Identical copy of a kept recording,InfantCry-DBL,Dunstan-Core,1
3,Identical copy of a kept recording,InfantCry-DBL,InfantCry-1214,196
4,Identical recording with conflicting labels,ICSD,Real Strong,4
5,Identical recording with conflicting labels,InfantCry-DBL,Dunstan-Core,3
6,Identical recording with conflicting labels,InfantCry-DBL,InfantCry-1214,3
7,InfantCry-DBL copy of a Donate-a-Cry recording,InfantCry-DBL,InfantCry-1214,13


# 4. Label Mapping

Original labels are converted into two targets: **Cry / Non-Cry** and **Hunger / Pain / Discomfort / Tired / Burping**. Separate columns allow recordings to support either or both tasks.

- ICSD background and snoring are Non-Cry; Nonspeech7k `crying` is excluded because infant origin is unconfirmed.
- OSF Acute and Residual Pain are both mapped to Pain.
- DBL `eairh` is provisionally mapped to Pain; this choice remains open to comparison.
- Chillanto contributes Hunger and Pain to cause classification. It is excluded from Cry Detection for now because its audio overlaps ICSD synthetic foreground material.

The cells below apply these rules and display the mapping table.

In [ ]:
CRY_DETECTION_MAPPING = {
    "ICSD": {
        "Infantcry": "Cry",
        "Snoring": "Non-Cry",
        # Materials Background
        "absence": "Non-Cry",
        "cooking": "Non-Cry",
        "dishwashing": "Non-Cry",
        "eating": "Non-Cry",
        "music": "Non-Cry",
        "noise": "Non-Cry",
        "other": "Non-Cry",
        "social_activity": "Non-Cry",
        "speech": "Non-Cry",
        "vacuum_cleaner": "Non-Cry",
        "watching_tv": "Non-Cry",
        "working": "Non-Cry",
    },
    "Donate-a-Cry": {
        "hungry": "Cry",
        "belly_pain": "Cry",
        "discomfort": "Cry",
        "tired": "Cry",
        "burping": "Cry",
    },
    "OSF": {
        "Discomfort": "Cry",
        "Acute Pain": "Cry",
        "Residual Pain": "Cry",
    },
    "InfantCry-DBL": {
        "neh": "Cry",
        "eairh": "Cry",
        "heh": "Cry",
        "owh": "Cry",
        "eh": "Cry",
    },
    "Nonspeech7k": {
        "breath": "Non-Cry",
        "cough": "Non-Cry",
        "laugh": "Non-Cry",
        "screaming": "Non-Cry",
        "sneeze": "Non-Cry",
        "yawn": "Non-Cry",
        # "crying" is excluded (not documented as infant cries)
    },
        "Baby Chillanto": {
        # hunger and pain are excluded for now (decision pending)
    },
}

CRY_CAUSE_MAPPING = {
    "Donate-a-Cry": {
        "hungry": "Hunger",
        "belly_pain": "Pain",
        "discomfort": "Discomfort",
        "tired": "Tired",
        "burping": "Burping",
    },
    "OSF": {
        "Discomfort": "Discomfort",
        "Acute Pain": "Pain",
        "Residual Pain": "Pain",
    },
    "InfantCry-DBL": {
        "neh": "Hunger",
        "eairh": "Pain",        # provisional
        "heh": "Discomfort",
        "owh": "Tired",
        "eh": "Burping",
    },
    "Baby Chillanto": {
        "hunger": "Hunger",
        "pain": "Pain",
    },
}

# Labels intentionally left out of Cry Detection, with the reason
EXCLUDED_LABELS = {
    ("Nonspeech7k", "crying"): "Not documented as infant cries",
    ("Baby Chillanto", "hunger"): "Pending decision (audio is part of ICSD synthetic foreground)",
    ("Baby Chillanto", "pain"): "Pending decision (audio is part of ICSD synthetic foreground)",
}


mapped_df = clean_audio_df.copy()

mapped_df["binary_label"] = [
    CRY_DETECTION_MAPPING[dataset].get(label)
    for dataset, label in zip(mapped_df["dataset"], mapped_df["original_label"])
]

mapped_df["cause_label"] = [
    CRY_CAUSE_MAPPING.get(dataset, {}).get(label)
    for dataset, label in zip(mapped_df["dataset"], mapped_df["original_label"])
]

print("Cry Detection:")
print(mapped_df["binary_label"].value_counts(dropna=False))
print("\nCry Cause:")
print(mapped_df["cause_label"].value_counts(dropna=False))

Cry Detection:
binary_label
Non-Cry    94158
Cry         7518
None        2538
Name: count, dtype: int64

Cry Cause:
cause_label
None          101576
Hunger           917
Tired            678
Pain             452
Discomfort       366
Burping          225
Name: count, dtype: int64


In [ ]:
mapping_rows = []

for dataset, labels in CRY_DETECTION_MAPPING.items():
    for original, binary in labels.items():
        mapping_rows.append({
            "Dataset": dataset,
            "Original Label": original,
            "Cry Detection": binary,
            "Cry Cause": CRY_CAUSE_MAPPING.get(dataset, {}).get(original, "—"),
        })

for (dataset, label), reason in EXCLUDED_LABELS.items():
    mapping_rows.append({
        "Dataset": dataset,
        "Original Label": label,
        "Cry Detection": f"Excluded — {reason}",
        "Cry Cause": CRY_CAUSE_MAPPING.get(dataset, {}).get(label, "—"),
    })

mapping_table = pd.DataFrame(mapping_rows)
display(mapping_table)

,Dataset,Original Label,Cry Detection,Cry Cause
0,ICSD,Infantcry,Cry,—
1,ICSD,Snoring,Non-Cry,—
2,ICSD,absence,Non-Cry,—
3,ICSD,cooking,Non-Cry,—
4,ICSD,dishwashing,Non-Cry,—
5,ICSD,eating,Non-Cry,—
6,ICSD,music,Non-Cry,—
7,ICSD,noise,Non-Cry,—
8,ICSD,other,Non-Cry,—
9,ICSD,social_activity,Non-Cry,—


## Cry Detection Distribution

Cry and Non-Cry counts are displayed overall and by source to show class imbalance and dataset contributions. ICSD Background is shown separately and in full; these are recording counts before clipping exclusions and background sampling.

In [ ]:
binary_df = mapped_df[mapped_df["binary_label"].notna()].copy()

# Show ICSD Strong and ICSD Materials Background as separate sources
binary_df["source"] = binary_df["dataset"]
binary_df.loc[binary_df["subset"] == "Materials Background", "source"] = "ICSD Materials Background"

binary_distribution = (
    binary_df["binary_label"]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)
binary_distribution["Percentage"] = (
    binary_distribution["Count"] / binary_distribution["Count"].sum() * 100
).round(2)

print("Cry Detection — overall")
display(binary_distribution)

print("Cry Detection — by source")
display(pd.crosstab(binary_df["source"], binary_df["binary_label"], margins=True, margins_name="Total"))

Cry Detection — overall


,Class,Count,Percentage
0,Non-Cry,94158,92.61
1,Cry,7518,7.39


Cry Detection — by source


binary_label,Cry,Non-Cry,Total
source,,,
Donate-a-Cry,457,0,457
ICSD,5422,5380,10802
ICSD Materials Background,0,83760,83760
InfantCry-DBL,1335,0,1335
Nonspeech7k,0,5018,5018
OSF,304,0,304
Total,7518,94158,101676


## Cry Cause Distribution

Cause counts and percentages are displayed overall and by dataset. This shows both class imbalance and whether a class is supplied mainly by one source.

In [ ]:
cause_df = mapped_df[mapped_df["cause_label"].notna()].copy()

cause_distribution = (
    cause_df["cause_label"]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)
cause_distribution["Percentage"] = (
    cause_distribution["Count"] / cause_distribution["Count"].sum() * 100
).round(2)

print("Cry Cause — overall")
display(cause_distribution)

print("Cry Cause — number of recordings by dataset")
display(pd.crosstab(cause_df["cause_label"], cause_df["dataset"], margins=True, margins_name="Total"))

print("Cry Cause — share of each class coming from each dataset (%)")
display(
    (pd.crosstab(cause_df["cause_label"], cause_df["dataset"], normalize="index") * 100).round(1)
)

Cry Cause — overall


,Class,Count,Percentage
0,Hunger,917,34.76
1,Tired,678,25.70
2,Pain,452,17.13
3,Discomfort,366,13.87
4,Burping,225,8.53


Cry Cause — number of recordings by dataset


dataset,Baby Chillanto,Donate-a-Cry,InfantCry-DBL,OSF,Total
cause_label,,,,,
Burping,0,8,217,0,225
Discomfort,0,27,129,210,366
Hunger,350,382,185,0,917
Pain,192,16,150,94,452
Tired,0,24,654,0,678
Total,542,457,1335,304,2638


Cry Cause — share of each class coming from each dataset (%)


dataset,Baby Chillanto,Donate-a-Cry,InfantCry-DBL,OSF
cause_label,,,,
Burping,0.0,3.6,96.4,0.0
Discomfort,0.0,7.4,35.2,57.4
Hunger,38.2,41.7,20.2,0.0
Pain,42.5,3.5,33.2,20.8
Tired,0.0,3.5,96.5,0.0


# 5. Audio Preprocessing (Steps 1–5)

Every file goes through four processing steps and one quality check before it is cut into windows:

1. **Mono, 16 kHz** — one channel and one sample rate for all datasets.
2. **16 → 8 → 16 kHz round trip** — removes everything above 4 kHz from all datasets.
3. **Peak normalisation to 1.0** — reduces recording-gain differences.
4. **Edge-silence trimming at −70 dB** — cry recordings only.
5. **Quality check by listening** — no file is excluded.

The cells below only define the functions; audio is processed when feature extraction calls `prepare_audio` (Section 8). The evidence behind each setting is in Appendix A.

## Step 1 — Mono and 16 kHz

Every file is converted to one channel and resampled to 16 kHz, so all datasets enter the pipeline in the same format.

**16 kHz.** Mpia (2025) and Tawfik (2026) both load audio at 16 kHz mono, and the pretrained model we may use later (YAMNet) requires mono audio at 16 kHz. It also keeps frequencies up to 8 kHz, above the cry harmonics (up to about 3 kHz, Mekhfioui 2025).

**Mono by channel average.** Averaging the channels is how librosa converts stereo to mono. It fails only if the channels cancel each other; the check in Appendix A.1 shows this does not happen here.

In [ ]:
TARGET_SR = 16000


def load_mono_16k(path):
    """Load an audio file as mono (channel average) at 16 kHz, float32."""
    audio, sr = sf.read(path, always_2d=True, dtype="float32")
    audio = audio.mean(axis=1)

    if sr != TARGET_SR:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=TARGET_SR)

    return audio

## Step 2 — Bandwidth Harmonisation

Donate-a-Cry is recorded at 8 kHz, so after resampling to 16 kHz it contains nothing above 4 kHz, while the other datasets do. A classifier can use that difference to recognise the **source** of a recording instead of the cry type (in an earlier test, the source could be predicted from the features with a balanced accuracy of about 0.89 with DBL and 0.96 without it).

**Decision: no round trip.** No model and none of the reviewed studies require a 16 → 8 → 16 kHz round trip, and it changes the audio of every dataset. Mpia (2025) and Tawfik (2026) use plain 16 kHz mono. Instead, the audio stays as it is (each dataset is only resampled to 16 kHz in Step 1), and the features are limited to 4 kHz when they are computed (Section 8):

- mel and MFCC (and delta): `fmax = 4000` (as before);
- centroid, bandwidth, roll-off, flatness, spectral contrast, chroma and RMS: only the STFT bins up to 4 kHz are used;
- ZCR: computed on the window with everything above 4 kHz removed in the frequency domain.

The F0 and periodicity features are unchanged (cry F0 is below 2 kHz).

No study tested this on infant cries, so it is our own decision. **Status: provisional.** The source-prediction test must be repeated on the new features; if the source is still clearly predictable, the alternatives are a full round trip or a 4 kHz low-pass filter on the audio.

In [ ]:
# Step 2 has no audio function: the audio is not modified.
# The 4 kHz limit is applied when the features are computed (Section 8, `band_limit` and `window_features`).

## Step 3 — Peak Normalisation

After Step 1, each file is scaled so its highest absolute sample equals 1.0, which reduces recording-gain differences between sources. It is applied after resampling, because resampling can change the peak. A target of 0.95 instead of 1.0 makes no difference for a Random Forest. The weakness is that a single click sets the scale for the whole file; the effect on loudness (RMS) per dataset is still to be measured.

In [ ]:
def peak_normalize(audio):
    """Scale the signal so its highest absolute sample equals 1.0."""
    peak = np.max(np.abs(audio))
    if peak == 0:
        return audio          # silent file: nothing to scale
    return audio / peak


def load_audio(path):
    """Steps 1 and 3: mono, 16 kHz, peak-normalized (the 4 kHz limit is applied to the features)."""
    audio = load_mono_16k(path)
    audio = peak_normalize(audio)
    return audio

## Step 4 — Edge Silence Trimming

Silence in a cry recording carries the cry label without containing cry, so it
is removed from the beginning and end of cry recordings before segmentation.

**From the literature.** Özseven (2023) and Rashid (2023) remove leading and
trailing silence; Mittal (2016), Sharma (2015), and Ji (2021) detect cry
boundaries with short-time energy, used together with zero-crossing rate.
Silence inside a cry is kept: studies that trim cry recordings for
classification remove only edge silence, and Yu & Li (2025), who join cry
segments after removing silence, do so for cry detection. Non-cry recordings
are not trimmed, because silence is a valid non-cry example (Yu & Li 2025).
No study reports a silence threshold, so it is set from the data below and
checked by listening, as Özseven (2023) and TinyML (2025) verified recordings by
ear.

In [ ]:
FRAME_LENGTH = int(0.025 * TARGET_SR)     # 25 ms
HOP_LENGTH_VAD = int(0.010 * TARGET_SR)   # 10 ms
CRY_DATASETS = ["Donate-a-Cry", "OSF", "InfantCry-DBL", "Baby Chillanto"]


def frame_energy_db(audio):
    """Short-time energy of each frame in dB (0 dB = full scale after peak normalization)."""
    rms = librosa.feature.rms(y=audio, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH_VAD)[0]
    return 20 * np.log10(rms + 1e-10)


def frame_zcr(audio):
    """Zero-crossing rate of each frame (share of sign changes)."""
    return librosa.feature.zero_crossing_rate(
        audio, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH_VAD
    )[0]

In [ ]:
SILENCE_DB = -70     # lower side of the OSF valley (Appendix A.3)


def active_bounds(audio):
    """First and last sample of the audio above SILENCE_DB (frame energy)."""
    db = frame_energy_db(audio)
    active = np.where(db > SILENCE_DB)[0]
    if len(active) == 0:
        return 0, 0
    start = active[0] * HOP_LENGTH_VAD
    end = min(len(audio), active[-1] * HOP_LENGTH_VAD + FRAME_LENGTH)
    return start, end


def trim_edge_silence(audio):
    """Remove leading and trailing silence; keep everything in between."""
    start, end = active_bounds(audio)
    return audio[start:end]


def prepare_audio(path, is_cry):
    """Steps 1–4: mono, 16 kHz, 4 kHz bandwidth, peak-normalized, edge silence trimmed (cry only)."""
    audio = load_audio(path)
    return trim_edge_silence(audio) if is_cry else audio

**Step 4 summary.** Leading and trailing frames below −70 dB are removed from
cry recordings before segmentation (`prepare_audio`). The threshold comes from
the OSF valley and was confirmed by listening; zero-crossing rate and a
minimum-duration rule are not used. Non-cry recordings are not trimmed.

## Step 5 — Quality Check

Recordings were checked by listening, as in Özseven (2023), Rashid (2023), and
TinyML (2025). Some Baby Chillanto `pain` segments are clipped, but the cry
remains clearly audible, so no file is excluded. No reviewed infant-cry study
measures or corrects clipping, so no clipping-based step is applied.

Unlike Özseven (2023), who kept only recordings without environmental noise,
noisy recordings are kept, because the device operates in homes; only the
clipped segments were checked by listening.

# 6. Train / Validation / Test Split

### Inspect Background Recording Groups

Background files are segments of longer recordings. The code checks their filename patterns to identify which segments share a source.

For SINS names (`DevNode<n>_ex<k>_<segment>`), `ex` identifies the same recording interval across microphones. Other files use the name before the segment number. These groups help keep related background audio together.

In [ ]:
bg = mapped_df[mapped_df["subset"] == "Materials Background"].copy()
names = bg["audio_path"].apply(os.path.basename).str.replace(".wav", "", regex=False)

# SINS: DevNode<node>_ex<recording>_<segment>
sins = names.str.extract(r"^DevNode(\d+)_ex(\d+)_(\d+)$")
# MUSAN-style: <recording>_<segment>
other = names.str.extract(r"^(.+)_(\d+)$")

bg["node"] = sins[0]
bg["ex"] = sins[1]
bg["recording"] = np.where(sins[1].notna(), "SINS_ex" + sins[1], other[0])

print("Names matching neither pattern:", bg["recording"].isna().sum())

# 1. Does the same ex number appear on more than one node?
s = bg[bg["ex"].notna()]
print("\n1) Number of nodes per ex number:")
print(s.groupby("ex")["node"].nunique().value_counts().sort_index())

# 2. Does an ex number ever have more than one class?
print("\n2) ex numbers with more than one class:",
      (s.groupby("ex")["original_label"].nunique() > 1).sum())

# 3. Recordings and segments per class
print("\n3) Recordings per class:")
display(
    bg.groupby("original_label")
      .agg(files=("audio_path", "size"), recordings=("recording", "nunique"))
      .assign(segments_per_recording=lambda d: (d["files"] / d["recordings"]).round(1))
)

Names matching neither pattern: 0

1) Number of nodes per ex number:
node
4    266
Name: count, dtype: int64

2) ex numbers with more than one class: 0

3) Recordings per class:


,files,recordings,segments_per_recording
original_label,,,
absence,18805,40,470.1
cooking,5124,13,394.2
dishwashing,1424,10,142.4
eating,2308,13,177.5
music,3600,641,5.6
noise,1831,493,3.7
other,2060,118,17.5
social_activity,4944,21,235.4
speech,5400,422,12.8


### Assign the Splits

Files are split by infant or recording, so no group appears in more than one
part. **20% of the groups form a fixed test set**, used once at the end; the
remaining 80% are divided into **5 folds** for cross-validation, so every
training file serves once as validation (Tawfik 2026: infant-disjoint
group-stratified 5-fold CV; scikit-learn User Guide 3.1: "A test set should
still be held out for final evaluation, but the validation set is no longer
needed when doing CV"). `StratifiedGroupKFold` balances dataset and original
label, with one fixed seed (42).

| Dataset | Group | Test |
|---|---|---|
| ICSD Strong | file | official test split |
| Nonspeech7k | File ID | official test split (the 6 File IDs found in both official splits go to train/validation) |
| ICSD Background | source recording | 20% of recordings |
| Donate-a-Cry | app installation | 20% of groups |
| OSF | infant | 20% of groups |
| Baby Chillanto | recording number | 20% of groups |
| InfantCry-DBL Tier 2 | infant ID | 20% of groups |
| InfantCry-DBL Tier 1 | — | never in test or validation (no infant IDs; `fold = -1`) |

Columns: `split` is `test` or `trainval`; `fold` is 0–4 for training/validation
files, −1 for files always in training, and empty for test files. With 20% test
and 5 folds, each fold uses about 64% of the data for training and 16% for
validation. Counting infants per class showed that every cause class has at
least 22 infants in the test set; Donate-a-Cry `Burping` and `Pain` (7 infants
each) cannot be evaluated on their own at any test share, so they are evaluated
together with the same class from other datasets.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="The least populated class")

from sklearn.model_selection import StratifiedGroupKFold

N_FOLDS = 5
RANDOM_STATE = 42


def group_folds(part):
    """Fold number (0 to N_FOLDS-1) for each row; groups stay together, stratified by dataset and label."""
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    folds = pd.Series(-1, index=part.index)
    for k, (_, idx) in enumerate(sgkf.split(part, part["strata"], part["split_group"])):
        folds.iloc[idx] = k
    return folds


split_df = mapped_df.reset_index(drop=True).copy()
split_df["split"] = None
split_df["fold"] = -1
split_df["split_group"] = None
split_df["strata"] = split_df["dataset"] + "|" + split_df["original_label"]
basename = split_df["audio_path"].apply(os.path.basename)

# ---------- 1. ICSD Strong: official test; official train + validation -> trainval ----------
icsd_strong = split_df["dataset"].eq("ICSD") & split_df["subset"].ne("Materials Background")
split_df.loc[icsd_strong, "split_group"] = "ICSD:" + basename[icsd_strong]
split_df.loc[icsd_strong, "split"] = np.where(split_df.loc[icsd_strong, "orig_split"].eq("test"), "test", "trainval")

# ---------- 2. ICSD Materials Background: group by source recording ----------
bg = split_df["subset"].eq("Materials Background")
stem = basename[bg].str.replace(".wav", "", regex=False)
sins_ex = stem.str.extract(r"^DevNode\d+_ex(\d+)_\d+$")[0]
other_rec = stem.str.extract(r"^(.+)_\d+$")[0]
split_df.loc[bg, "split_group"] = np.where(sins_ex.notna(), "SINS:ex" + sins_ex, "BG:" + other_rec)

# ---------- 3. Nonspeech7k: official test; File IDs in both official splits -> trainval ----------
ns = split_df["dataset"].eq("Nonspeech7k")
split_df.loc[ns, "split_group"] = "NS:" + split_df.loc[ns, "source_id"]
ns_train_ids = set(split_df.loc[ns & split_df["orig_split"].eq("train"), "source_id"])
ns_test_ids = set(split_df.loc[ns & split_df["orig_split"].eq("test"), "source_id"])
shared_ids = ns_train_ids & ns_test_ids
ns_test = ns & split_df["orig_split"].eq("test") & ~split_df["source_id"].isin(shared_ids)
split_df.loc[ns, "split"] = np.where(ns_test[ns], "test", "trainval")

# ---------- 4. Cry datasets with an identifier ----------
dbl_t1 = split_df["dataset"].eq("InfantCry-DBL") & split_df["subset"].eq("Dunstan-Core")
cry_ids = split_df["dataset"].isin(["Donate-a-Cry", "OSF", "Baby Chillanto", "InfantCry-DBL"]) & ~dbl_t1
split_df.loc[cry_ids, "split_group"] = split_df.loc[cry_ids, "dataset"] + ":" + split_df.loc[cry_ids, "source_id"]

# ---------- 5. InfantCry-DBL Tier 1: always training (no infant IDs) ----------
split_df.loc[dbl_t1, "split"] = "trainval"
split_df.loc[dbl_t1, "split_group"] = "DBL-T1:" + basename[dbl_t1]

# ---------- 6. Test set: 20% of groups (one fold of 5) ----------
# Cry datasets: folds across classes (an infant can have several classes)
first_folds = group_folds(split_df[cry_ids])
split_df.loc[first_folds.index, "split"] = np.where(first_folds == 0, "test", "trainval")

# Background: each type split on its own (each recording has one type), so every type
# with at least 5 recordings reaches the test set and every fold
for _, part in split_df[bg].groupby("original_label"):
    first_folds = group_folds(part)
    split_df.loc[first_folds.index, "split"] = np.where(first_folds == 0, "test", "trainval")

# ---------- 7. Cross-validation folds inside trainval (DBL Tier 1 stays at -1) ----------
for mask in [icsd_strong, ns, cry_ids]:
    part = split_df[mask & split_df["split"].eq("trainval")]
    split_df.loc[part.index, "fold"] = group_folds(part)

for _, part in split_df[bg & split_df["split"].eq("trainval")].groupby("original_label"):
    split_df.loc[part.index, "fold"] = group_folds(part)

print(split_df["split"].value_counts())
print(split_df.loc[split_df["split"].eq("trainval"), "fold"].value_counts().sort_index())

split
trainval    86450
test        17764
Name: count, dtype: int64
fold
-1      333
 0    15425
 1    18374
 2    21335
 3    14435
 4    16548
Name: count, dtype: int64


### Inspect Split Counts

Files per split and per fold, by dataset and class, and the number of groups in
each part.

In [ ]:
split_table = split_df.assign(group=split_df["cause_label"].fillna(split_df["original_label"]))

print("Files per split")
display(pd.crosstab([split_table["dataset"], split_table["group"]], split_table["split"],
                    margins=True, margins_name="Total"))

print("Training/validation files per fold (-1 = always training)")
trainval = split_table[split_table["split"].eq("trainval")]
display(pd.crosstab([trainval["dataset"], trainval["group"]], trainval["fold"]))

print("Groups per split")
display(split_table.groupby(["dataset", "split"])["split_group"].nunique().unstack(fill_value=0))

Files per split


split                            test  trainval   Total
dataset        group                                   
Baby Chillanto Hunger              79       271     350
               Pain                22       170     192
Donate-a-Cry   Burping              1         7       8
               Discomfort           2        25      27
               Hunger              83       299     382
               Pain                 1        15      16
               Tired                6        18      24
ICSD           Infantcry          542      4880    5422
               Snoring            539      4841    5380
               absence           4685     14120   18805
               cooking            772      4352    5124
               dishwashing        344      1080    1424
               eating             604      1704    2308
               music              812      2788    3600
               noise              391      1440    1831
               other              400      1660    2060
               social_activity   1596      3348    4944
               speech            1123      4277    5400
               vacuum_cleaner     168       804     972
               watching_tv       3380     15268   18648
               working           1280     17364   18644
InfantCry-DBL  Burping             28       189     217
               Discomfort          13       116     129
               Hunger              19       166     185
               Pain                23       127     150
               Tired              110       544     654
Nonspeech7k    breath             154      1696    1850
               cough               70       632     702
               crying             192      1804    1996
               laugh              130      1143    1273
               screaming           69       594     663
               sneeze              30       236     266
               yawn                48       216     264
OSF            Discomfort          33       177     210
               Pain                15        79      94
Total                           17764     86450  104214

Training/validation files per fold (-1 = always training)


fold                            -1     0     1     2     3     4
dataset        group                                            
Baby Chillanto Hunger            0    55    76    24    43    73
               Pain              0    61    15    47    24    23
Donate-a-Cry   Burping           0     1     0     1     3     2
               Discomfort        0     6     3     3     8     5
               Hunger            0    59    58    65    55    62
               Pain              0     3     0     0     3     9
               Tired             0     7     1     5     4     1
ICSD           Infantcry         0   977   976   975   976   976
               Snoring           0   968   968   969   968   968
               absence           0   824  2408  6784    20  4084
               cooking           0   956   872  1176  1204   144
               dishwashing       0   152   268   236   296   128
               eating            0   308   116   716   364   200
               music             0   581   632   634   515   426
               noise             0   366   323   302   178   271
               other             0   180   248   368   624   240
               social_activity   0   200  1600   408   904   236
               speech            0   693   822   805   989   968
               vacuum_cleaner    0   108   188   216   180   112
               watching_tv       0  3816  3252  6128  1312   760
               working           0  3844  3724   212  4208  5376
InfantCry-DBL  Burping          69    24    20    25    26    25
               Discomfort       57    11    15     8    15    10
               Hunger           71    18    25    25    16    11
               Pain             60    12    11    17    12    15
               Tired            76    95    89    88    99    97
Nonspeech7k    breath            0   120   903   119   216   338
               cough             0    91   161   156    93   131
               crying            0   430   286   401   323   364
               laugh             0   243    72   206   418   204
               screaming         0    69   121    75   197   132
               sneeze            0    49    34    60    50    43
               yawn              0    38    43    51    43    41
OSF            Discomfort        0    41    32    18    33    53
               Pain              0    19    12    12    16    20

Groups per split


split,test,trainval
dataset,,
Baby Chillanto,12,44
Donate-a-Cry,47,174
ICSD,1440,11184
InfantCry-DBL,178,1077
Nonspeech7k,401,1903
OSF,4,20


In [ ]:
# Infants (groups) per class, and how many would fall in test at each candidate share
cause = mapped_df[mapped_df["cause_label"].notna()]
groups = cause.groupby(["dataset", "cause_label"])["source_id"].nunique().rename("infants")
table = groups.to_frame()
for share in [0.10, 0.15, 0.20, 0.30]:
    table[f"test_{int(share * 100)}%"] = (table["infants"] * share).round(1)
display(table)

infants  test_10%  test_15%  test_20%  test_30%
dataset        cause_label                                                 
Baby Chillanto Hunger            33       3.3       5.0       6.6       9.9
               Pain              23       2.3       3.4       4.6       6.9
Donate-a-Cry   Burping            7       0.7       1.0       1.4       2.1
               Discomfort        16       1.6       2.4       3.2       4.8
               Hunger           205      20.5      30.8      41.0      61.5
               Pain               7       0.7       1.0       1.4       2.1
               Tired             16       1.6       2.4       3.2       4.8
InfantCry-DBL  Burping          146      14.6      21.9      29.2      43.8
               Discomfort        72       7.2      10.8      14.4      21.6
               Hunger           113      11.3      17.0      22.6      33.9
               Pain              89       8.9      13.4      17.8      26.7
               Tired            563      56.3      84.4     112.6     168.9
OSF            Discomfort        24       2.4       3.6       4.8       7.2
               Pain              24       2.4       3.6       4.8       7.2

> **⚠️ Sections 7 and 8 are the previous version and are being rewritten.
> Do not run them.**

# 7. Windows and File Selection

Every file is cut into **1-second windows without overlap**, after Steps 1–4.

| Data | Windows | Label |
|---|---|---|
| Cry datasets, Nonspeech7k | non-overlapping 1-s windows; the last partial second is dropped | label of the file |
| Files shorter than 1 s | one window, zero-padded (`PAD_MODE = "zero"`; `"none"` keeps the real length for the padding experiment) | label of the file |
| Baby Chillanto | each 1-s segment is one window | label of the file |
| ICSD Strong | ten 1-s segments per 10-s clip | `Cry` if any part of an infant-cry event falls inside the segment, otherwise `Non-Cry` |
| ICSD Background | ten 1-s segments per 10-s file | `Non-Cry` |

ICSD Strong is not edge-trimmed, so segment times stay aligned with the event
annotations. The segment rule follows the segment-based F1 used by ICSD
(Liu et al. 2024), where a 1-s segment is active when an event occurs within it
(Mesaros et al. 2016).

**Background files.** 350 files per type are drawn from train/validation, as
the 350 speech and music training clips in ICSD's competitive system; 360 files
per type (1 hour) are drawn from test, as ICSD's non-target test set (types
with fewer files are taken whole). Selection is random with seed 42.

In [ ]:
# Infant-cry events per ICSD Strong file: filename -> [(onset, offset), ...]
icsd_tables = [metadata["Real Train"], metadata["Synthetic Train"], real_val, synth_val,
               metadata["Real Test"], metadata["Synthetic Test"]]
icsd_events = pd.concat(icsd_tables, ignore_index=True)
icsd_events = icsd_events[icsd_events["event_label"] == "Infantcry"]

cry_events_by_file = (
    icsd_events.groupby("filename")[["onset", "offset"]]
    .apply(lambda g: list(zip(g["onset"], g["offset"])))
    .to_dict()
)
print("ICSD files with infant-cry events:", len(cry_events_by_file))

ICSD files with infant-cry events: 5424


In [ ]:
BG_TRAINVAL_PER_TYPE = 350   # ICSD competitive system: 350 speech and 350 music training clips
BG_TEST_PER_TYPE = 360       # ICSD non-target test set: 1 hour (360 x 10 s) per type

# Files used by at least one model (drops Nonspeech7k `crying`)
used = split_df[split_df["binary_label"].notna() | split_df["cause_label"].notna()]

is_bg = used["subset"].eq("Materials Background")


def sample_per_type(df, n):
    return (df.groupby("original_label", group_keys=False)[list(df.columns)]
              .apply(lambda g: g.sample(min(len(g), n), random_state=RANDOM_STATE)))


bg_selected = pd.concat([
    sample_per_type(used[is_bg & used["split"].eq("trainval")], BG_TRAINVAL_PER_TYPE),
    sample_per_type(used[is_bg & used["split"].eq("test")], BG_TEST_PER_TYPE),
])

work_df = pd.concat([used[~is_bg], bg_selected]).reset_index(drop=True)
work_df["events"] = work_df["audio_path"].map(
    lambda p: cry_events_by_file.get(os.path.basename(p), [])
)

print("Files to window:", len(work_df))
display(pd.crosstab([bg_selected["original_label"]], bg_selected["split"]))

Files to window: 26770


split,test,trainval
original_label,,
absence,360,350
cooking,360,350
dishwashing,344,350
eating,360,350
music,360,350
noise,360,350
other,360,350
social_activity,360,350
speech,360,350


In [ ]:
WIN = TARGET_SR          # 1 second
PAD_MODE = "zero"        # "zero" (baseline) or "none" (padding experiment)


def window_starts(n_samples):
    """Starts of non-overlapping 1-s windows; the last partial second is dropped.
    A file shorter than 1 s gives one window starting at 0."""
    if n_samples < WIN:
        return [0]
    return list(range(0, n_samples - WIN + 1, WIN))


def cut_window(audio, start):
    """One window; zero-padded to 1 s when PAD_MODE == 'zero'."""
    w = audio[start:start + WIN]
    if len(w) < WIN and PAD_MODE == "zero":
        w = np.pad(w, (0, WIN - len(w)))
    return w


def icsd_segment_label(start, events):
    """Cry if any part of an infant-cry event falls inside the segment (segment-based rule)."""
    seg_start, seg_end = start / TARGET_SR, (start + WIN) / TARGET_SR
    return "Cry" if any(on < seg_end and off > seg_start for on, off in events) else "Non-Cry"


def file_windows(row):
    """All windows of one file: list of (info, waveform)."""
    is_cry_file = row["dataset"] in CRY_DATASETS
    audio = prepare_audio(row["audio_path"], is_cry_file)       # Steps 1–4 (no trimming for ICSD)
    icsd_strong = row["dataset"] == "ICSD" and row["subset"] != "Materials Background"

    windows = []
    for s in window_starts(len(audio)):
        w = cut_window(audio, s)
        info = {
            "start_s": s / TARGET_SR,
            "win_s": min(WIN, len(audio) - s) / TARGET_SR,        # real audio in the window
            "binary_label": icsd_segment_label(s, row["events"]) if icsd_strong else row["binary_label"],
            "cause_label": row["cause_label"],
            "level_db": float(20 * np.log10(np.sqrt(np.mean(w ** 2)) + 1e-10)),
        }
        windows.append((info, w))
    return windows

In [ ]:
# [VERIFICATION] safe to delete — segment labels of one ICSD Strong file against its events
example = work_df[work_df["dataset"].eq("ICSD") & work_df["subset"].eq("Real Strong")
                  & work_df["events"].str.len().gt(0)].iloc[0]
print(os.path.basename(example["audio_path"]), "| events:", example["events"])
display(pd.DataFrame([info for info, _ in file_windows(example)])[["start_s", "binary_label"]])

strong_Infantcry_1.wav | events: [(3.807, 6.338)]


,start_s,binary_label
0,0.0,Non-Cry
1,1.0,Non-Cry
2,2.0,Non-Cry
3,3.0,Cry
4,4.0,Cry
5,5.0,Cry
6,6.0,Cry
7,7.0,Non-Cry
8,8.0,Non-Cry
9,9.0,Non-Cry


In [ ]:
# Silent windows in cry recordings (window energy below the Step 4 threshold)
cry_rows = work_df[work_df["dataset"].isin(CRY_DATASETS)]

rows = []
for _, row in tqdm(cry_rows.iterrows(), total=len(cry_rows), desc="Windowing cry files"):
    for info, _ in file_windows(row):
        rows.append({"dataset": row["dataset"], "label": row["cause_label"], **info})
cry_windows = pd.DataFrame(rows)

cry_windows["silent"] = cry_windows["level_db"] < SILENCE_DB
display(cry_windows.groupby(["dataset", "label"]).agg(
    windows=("silent", "size"),
    silent=("silent", "sum"),
    padded=("win_s", lambda s: (s < 1).sum()),
    lowest_db=("level_db", "min"),
).round(1))
print(f"Cry windows: {len(cry_windows)} | silent: {int(cry_windows['silent'].sum())}")

Windowing cry files:   0%|          | 0/2638 [00:00<?, ?it/s]

windows  silent  padded  lowest_db
dataset        label                                         
Baby Chillanto Hunger          350       0       0      -16.2
               Pain            192       0       0      -15.2
Donate-a-Cry   Burping          51       0       0      -44.5
               Discomfort      167       0       0      -49.3
               Hunger         2398       1       0      -71.1
               Pain            104       0       0      -54.6
               Tired           154       0       0      -57.7
InfantCry-DBL  Burping         387       0      20      -51.8
               Discomfort      199       0      14      -44.1
               Hunger          350       0      27      -65.7
               Pain            302       0       9      -37.9
               Tired          1024       0     108      -57.4
OSF            Discomfort     1226       0       0      -61.1
               Pain            546       0       0      -57.5

Cry windows: 7450 | silent: 1


# 8. Feature Extraction

Each 1-second window is described by the 213 hand-crafted features of Tawfik
(2026), the closest study to Mahd (InfantCry-DBL, infant-disjoint splits): 40
MFCC and 40 delta-MFCC, 12 chroma, 6 spectral-contrast bands, spectral centroid,
bandwidth, roll-off, flatness, zero-crossing rate, and RMS, each summarised by
its mean and standard deviation, plus F0 mean and standard deviation (voiced
frames), periodicity mean and standard deviation, and voiced fraction. Frames
use a 512-point FFT and a 160-sample hop, as in Tawfik (2026). Combining
several feature types follows the systematic review of Sirithepmontree et al.
(2026), where SVM performed better with multiple features than with one.

Two features are added from Mittal et al. (2016), who found that pain and
discomfort cries differ in the variability of F0 and energy: the coefficient of
variation (standard deviation divided by mean) of F0 and of frame energy.

The mel filterbank and spectral contrast stop at 4 kHz, the bandwidth left
after Step 2. F0 is estimated between 150 and 2000 Hz, covering typical cry F0
(200–800 Hz) and hyperphonation above 1000 Hz, which is linked to pain cries.

Features are saved to Drive in chunks of 2,000 files, so an interrupted run
resumes from the last saved chunk.

In [ ]:
import glob
from joblib import Parallel, delayed

N_FFT = 512              # Tawfik (2026)
HOP = 160                # Tawfik (2026): 10 ms
N_MELS = 64
N_MFCC = 40
FMAX_SPEC = 4000         # bandwidth after Step 2
F0_MIN, F0_MAX = 150, 2000
F0_FRAME = 1024
VOICED_THRESHOLD = 0.5

FEATURES_DIR = os.path.join(MAHD_ROOT, "features_svm_v1")
CHUNK_SIZE = 2000

META_COLS = ["audio_path", "dataset", "subset", "original_label", "source_id",
             "split", "fold", "split_group"]


def mean_std(name, m):
    m = np.atleast_2d(m)
    out = {}
    for i, row in enumerate(m):
        key = f"{name}{i}" if len(m) > 1 else name
        out[f"{key}_mean"] = float(row.mean())
        out[f"{key}_std"] = float(row.std())
    return out


def periodicity_track(w):
    """Peak of the normalised autocorrelation in the F0 lag range, per frame."""
    padded = np.pad(w, (F0_FRAME // 2, F0_FRAME // 2))
    frames = librosa.util.frame(padded, frame_length=F0_FRAME, hop_length=HOP)
    frames = frames - frames.mean(axis=0, keepdims=True)
    spec = np.fft.rfft(frames, n=2 * F0_FRAME, axis=0)
    ac = np.fft.irfft(np.abs(spec) ** 2, axis=0)[:F0_FRAME]
    ac = ac / (ac[0] + 1e-10)
    lo, hi = int(TARGET_SR / F0_MAX), int(TARGET_SR / F0_MIN) + 1
    return ac[lo:hi].max(axis=0)


def window_features(w):
    """215 features of one window: Tawfik (2026) set + Mittal (2016) variability."""
    S = np.abs(librosa.stft(w, n_fft=N_FFT, hop_length=HOP))
    mel = librosa.feature.melspectrogram(S=S ** 2, sr=TARGET_SR, n_mels=N_MELS, fmax=FMAX_SPEC)
    mfcc = librosa.feature.mfcc(S=librosa.power_to_db(mel), n_mfcc=N_MFCC)
    rms = librosa.feature.rms(S=S, frame_length=N_FFT)

    f = {}
    f.update(mean_std("mfcc", mfcc))
    f.update(mean_std("dmfcc", librosa.feature.delta(mfcc, mode="nearest")))
    f.update(mean_std("chroma", librosa.feature.chroma_stft(S=S ** 2, sr=TARGET_SR)))
    f.update(mean_std("contrast", librosa.feature.spectral_contrast(S=S, sr=TARGET_SR, fmin=125, n_bands=5)))
    f.update(mean_std("centroid", librosa.feature.spectral_centroid(S=S, sr=TARGET_SR)))
    f.update(mean_std("bandwidth", librosa.feature.spectral_bandwidth(S=S, sr=TARGET_SR)))
    f.update(mean_std("rolloff", librosa.feature.spectral_rolloff(S=S, sr=TARGET_SR)))
    f.update(mean_std("flatness", librosa.feature.spectral_flatness(S=S)))
    f.update(mean_std("zcr", librosa.feature.zero_crossing_rate(w, frame_length=N_FFT, hop_length=HOP)))
    f.update(mean_std("rms", rms))

    # Source features (Tawfik 2026)
    f0 = librosa.yin(w, fmin=F0_MIN, fmax=F0_MAX, sr=TARGET_SR, frame_length=F0_FRAME, hop_length=HOP)
    periodicity = periodicity_track(w)
    n = min(len(f0), len(periodicity))
    f0, periodicity = f0[:n], periodicity[:n]
    voiced = periodicity >= VOICED_THRESHOLD
    f0_voiced = f0[voiced]

    f["f0_mean"] = float(f0_voiced.mean()) if voiced.any() else 0.0
    f["f0_std"] = float(f0_voiced.std()) if voiced.any() else 0.0
    f["periodicity_mean"] = float(periodicity.mean())
    f["periodicity_std"] = float(periodicity.std())
    f["voiced_fraction"] = float(voiced.mean())

    # Variability (Mittal 2016): coefficient of variation of F0 and of energy
    f["f0_cv"] = f["f0_std"] / f["f0_mean"] if f["f0_mean"] > 0 else 0.0
    f["rms_cv"] = float(rms.std() / (rms.mean() + 1e-10))

    return {k: float(np.nan_to_num(v)) for k, v in f.items()}


def process_file(row):
    """Features of every window of one file. Returns (rows, error)."""
    try:
        rows = []
        for info, w in file_windows(row):
            rows.append({**{k: row[k] for k in META_COLS}, **info, **window_features(w)})
        return rows, None
    except Exception as e:
        return [], f"{row['audio_path']}: {e}"


# Quick check on one window
test_w = cut_window(load_audio(work_df.iloc[0]["audio_path"]), 0)
print("Number of features:", len(window_features(test_w)))

NameError: name 'SR' is not defined

### Extract and Save Feature Chunks

The prepared files are processed in batches of 2,000 input files. Each completed batch is saved as a Parquet file, while processing errors are logged separately. Existing chunks are skipped to resume interrupted runs; clear them if inputs or settings change.

In [ ]:
import glob
from joblib import Parallel, delayed

N_FFT = 512              # Tawfik (2026)
HOP = 160                # Tawfik (2026): 10 ms
N_MELS = 64
N_MFCC = 40
FMAX_SPEC = 4000         # bandwidth after Step 2
F0_MIN, F0_MAX = 150, 2000
F0_FRAME = 1024
VOICED_THRESHOLD = 0.5

FEATURES_DIR = os.path.join(MAHD_ROOT, "features_svm_v1")
CHUNK_SIZE = 2000

META_COLS = ["audio_path", "dataset", "subset", "original_label", "source_id",
             "split", "fold", "split_group"]


def mean_std(name, m):
    m = np.atleast_2d(m)
    out = {}
    for i, row in enumerate(m):
        key = f"{name}{i}" if len(m) > 1 else name
        out[f"{key}_mean"] = float(row.mean())
        out[f"{key}_std"] = float(row.std())
    return out


def periodicity_track(w):
    """Peak of the normalised autocorrelation in the F0 lag range, per frame."""
    padded = np.pad(w, (F0_FRAME // 2, F0_FRAME // 2))
    frames = librosa.util.frame(padded, frame_length=F0_FRAME, hop_length=HOP)
    frames = frames - frames.mean(axis=0, keepdims=True)
    spec = np.fft.rfft(frames, n=2 * F0_FRAME, axis=0)
    ac = np.fft.irfft(np.abs(spec) ** 2, axis=0)[:F0_FRAME]
    ac = ac / (ac[0] + 1e-10)
    lo, hi = int(TARGET_SR / F0_MAX), int(TARGET_SR / F0_MIN) + 1
    return ac[lo:hi].max(axis=0)


def window_features(w):
    """215 features of one window: Tawfik (2026) set + Mittal (2016) variability."""
    S = np.abs(librosa.stft(w, n_fft=N_FFT, hop_length=HOP))
    mel = librosa.feature.melspectrogram(S=S ** 2, sr=TARGET_SR, n_mels=N_MELS, fmax=FMAX_SPEC)
    mfcc = librosa.feature.mfcc(S=librosa.power_to_db(mel), n_mfcc=N_MFCC)
    rms = librosa.feature.rms(S=S, frame_length=N_FFT)

    f = {}
    f.update(mean_std("mfcc", mfcc))
    f.update(mean_std("dmfcc", librosa.feature.delta(mfcc, mode="nearest")))
    f.update(mean_std("chroma", librosa.feature.chroma_stft(S=S ** 2, sr=TARGET_SR)))
    f.update(mean_std("contrast", librosa.feature.spectral_contrast(S=S, sr=TARGET_SR, fmin=125, n_bands=5)))
    f.update(mean_std("centroid", librosa.feature.spectral_centroid(S=S, sr=TARGET_SR)))
    f.update(mean_std("bandwidth", librosa.feature.spectral_bandwidth(S=S, sr=TARGET_SR)))
    f.update(mean_std("rolloff", librosa.feature.spectral_rolloff(S=S, sr=TARGET_SR)))
    f.update(mean_std("flatness", librosa.feature.spectral_flatness(S=S)))
    f.update(mean_std("zcr", librosa.feature.zero_crossing_rate(w, frame_length=N_FFT, hop_length=HOP)))
    f.update(mean_std("rms", rms))

    # Source features (Tawfik 2026)
    f0 = librosa.yin(w, fmin=F0_MIN, fmax=F0_MAX, sr=TARGET_SR, frame_length=F0_FRAME, hop_length=HOP)
    periodicity = periodicity_track(w)
    n = min(len(f0), len(periodicity))
    f0, periodicity = f0[:n], periodicity[:n]
    voiced = periodicity >= VOICED_THRESHOLD
    f0_voiced = f0[voiced]

    f["f0_mean"] = float(f0_voiced.mean()) if voiced.any() else 0.0
    f["f0_std"] = float(f0_voiced.std()) if voiced.any() else 0.0
    f["periodicity_mean"] = float(periodicity.mean())
    f["periodicity_std"] = float(periodicity.std())
    f["voiced_fraction"] = float(voiced.mean())

    # Variability (Mittal 2016): coefficient of variation of F0 and of energy
    f["f0_cv"] = f["f0_std"] / f["f0_mean"] if f["f0_mean"] > 0 else 0.0
    f["rms_cv"] = float(rms.std() / (rms.mean() + 1e-10))

    return {k: float(np.nan_to_num(v)) for k, v in f.items()}


def process_file(row):
    """Features of every window of one file. Returns (rows, error)."""
    try:
        rows = []
        for info, w in file_windows(row):
            rows.append({**{k: row[k] for k in META_COLS}, **info, **window_features(w)})
        return rows, None
    except Exception as e:
        return [], f"{row['audio_path']}: {e}"


# Quick check on one window
test_w = cut_window(load_audio(work_df.iloc[0]["audio_path"]), 0)
print("Number of features:", len(window_features(test_w)))

Number of features: 215


In [ ]:
FEATURES_DIR = os.path.join(MAHD_ROOT, "features_cause_v1")   # cry-cause files only
os.makedirs(FEATURES_DIR, exist_ok=True)
tasks = work_df[work_df["cause_label"].notna()].to_dict("records")
print("Cry-cause files:", len(tasks))
n_chunks = int(np.ceil(len(tasks) / CHUNK_SIZE))
error_log = os.path.join(FEATURES_DIR, "errors.txt")

for c in range(n_chunks):
    chunk_path = os.path.join(FEATURES_DIR, f"chunk_{c:03d}.parquet")
    if os.path.exists(chunk_path):
        continue

    chunk = tasks[c * CHUNK_SIZE:(c + 1) * CHUNK_SIZE]
    results = Parallel(n_jobs=-1)(delayed(process_file)(r) for r in chunk)

    rows = [w for windows, _ in results for w in windows]
    errors = [e for _, e in results if e]

    pd.DataFrame(rows).to_parquet(chunk_path, index=False)
    if errors:
        with open(error_log, "a") as f:
            f.write("\n".join(errors) + "\n")

    print(f"Chunk {c + 1}/{n_chunks}: {len(chunk)} files -> {len(rows)} windows, {len(errors)} errors")

Cry-cause files: 2638
Chunk 1/2: 2000 files -> 6777 windows, 0 errors
Chunk 2/2: 638 files -> 673 windows, 0 errors


# Appendix A — Evidence for the Preprocessing Decisions (optional, slow)

These cells measure the data behind Steps 1–4, plus the audio inventory and durations. **The pipeline does not need them**: skip them unless you want to reproduce or show the evidence. Some read every recording and take several minutes.

## A.0 Audio Inventory and Durations

Each file's sample rate, channel count, duration, and format are read without
loading the audio. The inventory shows why Step 1 is needed (seven sample rates,
stereo files in InfantCry-DBL), and the durations support the 1-second window
(InfantCry-DBL has a median of about 1.7 s).

In [ ]:
def inspect_audio(path):
    try:
        info = sf.info(path)

        return {
            "sample_rate": info.samplerate,
            "channels": info.channels,
            "duration_sec": info.duration,
            "subtype": info.subtype,
            "status": "OK"
        }

    except Exception as e:
        return {
            "sample_rate": np.nan,
            "channels": np.nan,
            "duration_sec": np.nan,
            "subtype": None,
            "status": f"ERROR: {e}"
        }

In [ ]:
audio_inventory = []

for _, row in tqdm(
    mapped_df.iterrows(),
    total=len(mapped_df),
    desc="Inspecting audio files"
):

    info = inspect_audio(row["audio_path"])

    audio_inventory.append({
        "audio_path": row["audio_path"],
        "dataset": row["dataset"],
        "subset": row["subset"],
        "original_label": row["original_label"],
        "source_id": row["source_id"],
        "source_id_type": row["source_id_type"],
        "binary_label": row["binary_label"],
        "cause_label": row["cause_label"],
        **info
    })

audio_inventory_df = pd.DataFrame(audio_inventory)

print("Total files inspected:", len(audio_inventory_df))

### Duration by Dataset and Class

The duration summary reports the percentage of files shorter than 1, 2, and 3 seconds. These values show how many files are shorter than each candidate window length, especially in short-recording classes. Files shorter than 1 s are zero-padded (Section 7). ICSD Background is excluded because its files are 10 seconds long.

In [ ]:
# Duration per dataset and class (ICSD Materials Background excluded: all 10 s)
durations = audio_inventory_df[audio_inventory_df["subset"] != "Materials Background"].copy()
durations["group"] = durations["cause_label"].fillna(durations["original_label"])

duration_summary = (
    durations
    .groupby(["dataset", "group"])["duration_sec"]
    .agg(
        files="size",
        p10=lambda s: s.quantile(0.10),
        median="median",
        under_1s=lambda s: (s < 1).mean() * 100,
        under_2s=lambda s: (s < 2).mean() * 100,
        under_3s=lambda s: (s < 3).mean() * 100,
        total_seconds="sum",
    )
    .round(2)
)

display(duration_summary)

### Total Cry Audio

Total duration of the cry recordings, per dataset and per class, from the
original files (before edge-silence trimming). ICSD `Infantcry` is counted from
its annotated cry events, because its files are 10 s clips that contain cry only
in part.

In [ ]:
# Total cry audio: per dataset, per class, and overall (original durations, before trimming)
cry_inventory = audio_inventory_df[audio_inventory_df["dataset"].isin(CRY_DATASETS)].copy()
cry_inventory["class"] = cry_inventory["cause_label"].fillna(cry_inventory["original_label"])


def totals(df, by):
    t = df.groupby(by)["duration_sec"].agg(files="size", total_seconds="sum")
    t["total_minutes"] = t["total_seconds"] / 60
    return t.round(1)


display(totals(cry_inventory, "dataset"))
display(totals(cry_inventory, ["class", "dataset"]))
display(totals(cry_inventory, "class"))

total_s = cry_inventory["duration_sec"].sum()
print(f"Cry datasets: {len(cry_inventory)} files, {total_s:,.1f} s "
      f"({total_s / 60:,.1f} min, {total_s / 3600:.2f} h)")

# ICSD Infantcry: annotated cry events only (Cry Detection model)
icsd_tables = {"Real Train": metadata["Real Train"], "Synthetic Train": metadata["Synthetic Train"],
               "Real Validation": real_val, "Synthetic Validation": synth_val,
               "Real Test": metadata["Real Test"], "Synthetic Test": metadata["Synthetic Test"]}
icsd_cry = pd.concat([t.assign(part=name) for name, t in icsd_tables.items()], ignore_index=True)
icsd_cry = icsd_cry[icsd_cry["event_label"] == "Infantcry"]
icsd_cry["event_s"] = icsd_cry["offset"] - icsd_cry["onset"]
icsd_events = icsd_cry.groupby("part")["event_s"].agg(events="size", total_seconds="sum")
icsd_events["total_minutes"] = icsd_events["total_seconds"] / 60
display(icsd_events.round(1))
icsd_s = icsd_cry["event_s"].sum()
print(f"ICSD Infantcry events: {len(icsd_cry)} events, {icsd_s:,.1f} s ({icsd_s / 60:,.1f} min)")
print(f"All cry audio: {total_s + icsd_s:,.1f} s ({(total_s + icsd_s) / 3600:.2f} h)")

## A.1 Step 1 — Do the stereo channels cancel each other?

In [ ]:
# Inter-channel correlation of every stereo file
# ICSD is checked on a sample only: its authors converted every file to one channel.
non_icsd = all_audio_df[all_audio_df["dataset"] != "ICSD"]
#icsd_sample = all_audio_df[all_audio_df["dataset"] == "ICSD"].sample(200, random_state=42)

files_to_check = pd.concat([non_icsd])
channels = [sf.info(p).channels for p in tqdm(files_to_check["audio_path"], desc="Reading channels")]
files_to_check = files_to_check.assign(channels=channels)

print("ICSD sample, channel counts:", files_to_check.loc[files_to_check["dataset"] == "ICSD", "channels"].value_counts().to_dict())
print("Stereo files per dataset:")
display(files_to_check[files_to_check["channels"] == 2].groupby(["dataset", "subset"]).size())

stereo_paths = files_to_check.loc[files_to_check["channels"] == 2, "audio_path"].tolist()

correlations = []
for path in tqdm(stereo_paths, desc="Stereo files"):
    audio, _ = sf.read(path, always_2d=True, dtype="float32")
    left, right = audio[:, 0], audio[:, 1]
    if left.std() == 0 or right.std() == 0:
        correlations.append(np.nan)      # silent channel
    else:
        correlations.append(np.corrcoef(left, right)[0, 1])

stereo_check = pd.DataFrame({"audio_path": stereo_paths, "correlation": correlations})
stereo_check = stereo_check.merge(all_audio_df[["audio_path", "dataset", "subset"]], on="audio_path")

print("Stereo files:", len(stereo_check))
display(stereo_check.groupby(["dataset", "subset"])["correlation"].describe().round(3))
print("Files with negative correlation:", int((stereo_check["correlation"] < 0).sum()))
print("Files with one silent channel:", int(stereo_check["correlation"].isna().sum()))

## A.2 Step 2 — Energy above 4 kHz before and after the round trip

In [ ]:
from scipy.signal import welch

def frac_above_4k(y):
    f, p = welch(y, TARGET_SR, nperseg=1024)
    return p[f >= 4000].sum() / p.sum()

cause_sets = ["Baby Chillanto", "Donate-a-Cry", "InfantCry-DBL", "OSF"]
sample = (all_audio_df[all_audio_df["dataset"].isin(cause_sets)]
          .groupby(["dataset", "subset"], group_keys=False)
          .apply(lambda g: g.sample(min(len(g), 60), random_state=42)))

rows = []
for _, r in tqdm(sample.iterrows(), total=len(sample)):
    y = load_mono_16k(r["audio_path"])
    rows.append({"dataset": r["dataset"], "subset": r["subset"],
                 "above_4k": frac_above_4k(y)})
res = pd.DataFrame(rows)
display(res.groupby(["dataset", "subset"]).agg(
    median=("above_4k", "median"),
    p90=("above_4k", lambda s: s.quantile(0.9))).map("{:.2e}".format))

In [ ]:
chi = all_audio_df[all_audio_df["dataset"] == "Baby Chillanto"].copy()
chi["sr"] = [sf.info(p).samplerate for p in chi["audio_path"]]
display(pd.crosstab(chi["sr"], chi["original_label"]))

## A.3 Step 4 — Silence threshold and removed silence

In [ ]:
cry_files = all_audio_df[all_audio_df["dataset"].isin(CRY_DATASETS)]

In [ ]:
# Frame energy and zero-crossing rate in every cry recording (after Steps 1–3)
cry_files = all_audio_df[all_audio_df["dataset"].isin(CRY_DATASETS)]

frames = []
for _, row in tqdm(cry_files.iterrows(), total=len(cry_files), desc="Measuring frames"):
    audio = load_audio(row["audio_path"])
    frames.append(pd.DataFrame({
        "dataset": row["dataset"],
        "db": frame_energy_db(audio),
        "zcr": frame_zcr(audio),
    }))
frames = pd.concat(frames, ignore_index=True)

digital_silence = frames["db"] <= -100          # exact zeros
print("Frames:", len(frames), "| digital silence (exact zeros):", int(digital_silence.sum()))
frames = frames[~digital_silence]

In [ ]:
import matplotlib.colors
from scipy.ndimage import uniform_filter1d
from scipy.signal import find_peaks

BINS = np.arange(-100, 0.5, 1.0)     # 1-dB bins
MIN_PROMINENCE = 0.3                 # log10 frames: a valley at least ~2x lower than its peaks


def peaks_and_valleys(db_values):
    """Peaks and valleys of the smoothed log-histogram, with valley depth."""
    counts, edges = np.histogram(db_values, bins=BINS)
    centres = (edges[:-1] + edges[1:]) / 2
    log_counts = uniform_filter1d(np.log10(counts + 1), size=3)
    peaks, _ = find_peaks(log_counts, prominence=MIN_PROMINENCE)
    valleys, props = find_peaks(-log_counts, prominence=MIN_PROMINENCE)
    return centres[peaks], centres[valleys], props["prominences"]


fig, axes = plt.subplots(1, len(CRY_DATASETS), figsize=(5 * len(CRY_DATASETS), 3.5), sharey=True)
peak_rows = []
for ax, name in zip(axes, CRY_DATASETS):
    db = frames.loc[frames["dataset"] == name, "db"]
    p, v, depth = peaks_and_valleys(db)
    peak_rows.append({"dataset": name, "peaks_db": p.round(1).tolist(),
                      "valleys_db": v.round(1).tolist(), "valley_depth": depth.round(2).tolist()})
    ax.hist(db, bins=BINS)
    ax.set_yscale("log")
    ax.set_title(name)
    ax.set_xlabel("Frame energy (dB)")
axes[0].set_ylabel("Frames")
plt.tight_layout()
plt.show()

display(pd.DataFrame(peak_rows))
display(frames.groupby("dataset")["db"].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.5]).round(1))

In [ ]:
fig, axes = plt.subplots(1, len(CRY_DATASETS), figsize=(5 * len(CRY_DATASETS), 4), sharey=True)
for ax, name in zip(axes, CRY_DATASETS):
    d = frames[frames["dataset"] == name]
    ax.hist2d(d["db"], d["zcr"], bins=[60, 60], range=[[-100, 0], [0, 0.5]],
              cmap="viridis", norm=matplotlib.colors.LogNorm())
    ax.set_title(name)
    ax.set_xlabel("Frame energy (dB)")
axes[0].set_ylabel("Zero-crossing rate")
plt.tight_layout()
plt.show()

# ZCR of quiet vs louder frames, split at the OSF valley (the only dataset with a silence peak)
osf_valley = pd.DataFrame(peak_rows).set_index("dataset").loc["OSF", "valleys_db"][0]
frames["side"] = np.where(frames["db"] < osf_valley, "below OSF valley", "above OSF valley")
print("OSF valley:", osf_valley, "dB")
display(frames.groupby(["dataset", "side"])["zcr"].describe(percentiles=[0.25, 0.5, 0.75]).round(3))

**Silence threshold: −70 dB.** Only OSF has a silence peak (−78.5 dB) separate
from its cry peak (−15.5 dB), with a valley at −68.5 dB (depth 0.89, about eight
times lower than the peaks); its quiet frames have a zero-crossing rate of 0.000
(median), against 0.172 for the rest. InfantCry-DBL has a small quiet group below
−91.5 dB, and Donate-a-Cry has no valley. The threshold is set at −70 dB, on the
lower side of the OSF valley, so no frame above the valley is removed; one
threshold is used for all datasets.

**Zero-crossing rate is not used.** Where it separates quiet from cry frames
(OSF), energy already does; where energy does not (Donate-a-Cry, InfantCry-DBL),
neither does zero-crossing rate (median 0.178 vs 0.122; 0.172 vs 0.175).

Baby Chillanto shows two small groups at −52 and −40 dB with a zero-crossing
rate of zero, possibly the quiet floor of its original 8-bit recordings; they
are fewer than 1% of its frames, lie inside segments, and are above the
threshold, so they are kept.

In [ ]:
# Edge silence removed from every cry recording, and duration left after trimming
rows = []
for _, row in tqdm(cry_files.iterrows(), total=len(cry_files), desc="Measuring"):
    audio = load_audio(row["audio_path"])
    start, end = active_bounds(audio)
    rows.append({"audio_path": row["audio_path"], "dataset": row["dataset"],
                 "label": row["original_label"], "seconds": len(audio) / TARGET_SR,
                 "removed_start_s": start / TARGET_SR,
                 "removed_end_s": (len(audio) - end) / TARGET_SR,
                 "kept_s": (end - start) / TARGET_SR})

silence = pd.DataFrame(rows)
silence["removed_s"] = silence["removed_start_s"] + silence["removed_end_s"]

display(silence.groupby("dataset").agg(
    files=("audio_path", "size"),
    any_removed=("removed_s", lambda s: (s > 0).sum()),
    over_0_5s=("removed_s", lambda s: (s > 0.5).sum()),
    over_1s=("removed_s", lambda s: (s > 1.0).sum()),
    max_removed_s=("removed_s", "max"),
    fully_silent=("kept_s", lambda s: (s == 0).sum()),
    kept_under_250ms=("kept_s", lambda s: (s < 0.25).sum()),
    kept_under_400ms=("kept_s", lambda s: (s < 0.40).sum()),
    kept_under_500ms=("kept_s", lambda s: (s < 0.50).sum()),
    shortest_kept_s=("kept_s", "min"),
).round(2))

**Removed silence.** At −70 dB, edge silence is removed from 299 of 304 OSF recordings (294 lose more than 0.5 s; maximum 1.10 s), 303 of 457 Donate-a-Cry recordings (20 lose more than 0.5 s; maximum 3.32 s), and 142 of 1,551 InfantCry-DBL recordings (10 lose more than 0.5 s; maximum 0.70 s); nothing is removed from Baby Chillanto, and no recording is fully silent.

**No minimum-duration rule.** Published minimum durations are practical choices without a stated basis (ICSD: 250 ms; Rashid 2023: 500 ms, on non-cry sounds). After trimming, no cry recording is shorter than 250 ms; the shortest is 0.41 s (InfantCry-DBL), matching the minimum in the DBL README. Only Rashid's 500-ms limit would remove a handful of InfantCry-DBL files (see the duration table in Section 6), so no minimum-duration rule is applied.

In [ ]:
# Optional: listen to the removed parts, largest first
from IPython.display import Audio

DATASET_TO_CHECK = "Donate-a-Cry"   # "OSF", "InfantCry-DBL", "Donate-a-Cry"
MIN_REMOVED_S = 0.5

to_listen = (silence[(silence["dataset"] == DATASET_TO_CHECK) & (silence["removed_s"] > MIN_REMOVED_S)]
             .sort_values("removed_s", ascending=False))
print(f"{DATASET_TO_CHECK}: {len(to_listen)} recordings")

gap = np.zeros(int(0.3 * TARGET_SR))
for _, row in to_listen.iterrows():
    audio = load_audio(row["audio_path"])
    start, end = active_bounds(audio)
    print(f"{os.path.basename(row['audio_path'])} | {row['label']} | "
          f"start {row['removed_start_s']:.2f} s, end {row['removed_end_s']:.2f} s")
    display(Audio(np.concatenate([audio[:start], gap, audio[end:]]), rate=TARGET_SR))

In [9]:
import os
import warnings
warnings.filterwarnings("ignore")
# Safely handle Google Drive mounting
if not os.path.exists('/content/drive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print("Warning: Google Drive could not be mounted automatically.")
        print("Please click the 'Files' folder icon on the left sidebar and click 'Mount Drive'.")
        print(f"Error details: {e}")
else:
    print("Google Drive is already mounted.")

import glob
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score, balanced_accuracy_score, matthews_corrcoef

# 1. قراءة الميزات المحدثة (215 ميزة) من المجلد الجديد
MAHD_ROOT = "/content/drive/MyDrive/Mahd"
FEATURES_DIR = os.path.join(MAHD_ROOT, "features_cause_v1")

parquet_files = sorted(glob.glob(os.path.join(FEATURES_DIR, "chunk_*.parquet")))
features_df = pd.concat([pd.read_parquet(p) for p in parquet_files], ignore_index=True)

# 2. تحديد أعمدة الـ Metadata وأعمدة الـ 215 ميزة
META_COLS = [
    "audio_path", "dataset", "subset", "original_label", "source_id",
    "split", "fold", "split_group", "start_s", "win_s",
    "binary_label", "cause_label", "level_db"
]
feature_cols = [c for c in features_df.columns if c not in META_COLS]
print(f"Total windows: {len(features_df)} | Number of features: {len(feature_cols)}")

# 3. فصل الـ trainval فقط (شرط الليدر: ما أحد يلمس الـ test!)
trainval_df = features_df[features_df["split"] == "trainval"].copy().reset_index(drop=True)

le = LabelEncoder()
trainval_df["target"] = le.fit_transform(trainval_df["cause_label"])
classes = le.classes_
print("Classes:", list(classes))

# 4. دالة لتقييم أي مودل عبر الـ 5 Folds بنفس شروط الليدر (Window + Recording + Per-Dataset)
def evaluate_model_cv(model_name, make_model_fn, use_sample_weight=False):
    win_metrics, rec_metrics = [], []
    all_val_rows = []

    for k in range(5):
        # التدريب يشمل كل الـ folds ما عدا k (بما فيها fold == -1 الخاصة بـ DBL Tier 1)
        train_mask = (trainval_df["fold"] != k)
        val_mask   = (trainval_df["fold"] == k)

        tr = trainval_df[train_mask]
        va = trainval_df[val_mask].copy()

        # Scaling محسوب من الـ Train فقط لمنع الـ Leakage
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(tr[feature_cols])
        y_tr = tr["target"].values
        X_va = scaler.transform(va[feature_cols])
        y_va = va["target"].values

        model = make_model_fn()
        # MLPClassifier does not support sample_weight natively, so we bypass it if the model is MLP
        if use_sample_weight and not isinstance(model, MLPClassifier):
            sw = compute_sample_weight("balanced", y_tr)
            model.fit(X_tr, y_tr, sample_weight=sw)
        else:
            model.fit(X_tr, y_tr)

        # التوقع واحتمالات الفئات (للتجميع على مستوى التسجيل)
        preds = model.predict(X_va)
        probs = model.predict_proba(X_va)

        # أ) مقاييس مستوى النافذة (Window-level)
        win_metrics.append({
            "fold": k,
            "Macro-F1": f1_score(y_va, preds, average="macro"),
            "Balanced_Acc": balanced_accuracy_score(y_va, preds),
            "MCC": matthews_corrcoef(y_va, preds)
        })

        # ب) تجميع التوقعات على مستوى التسجيل الكامل (Recording-level) عبر متوسط الاحتمالات
        va["pred"] = preds
        for i, cls_name in enumerate(classes):
            va[f"prob_{i}"] = probs[:, i]
        all_val_rows.append(va)

        prob_cols = [f"prob_{i}" for i in range(len(classes))]
        rec_df = va.groupby("audio_path").agg(
            y_true=("target", "first"),
            dataset=("dataset", "first"),
            **{col: (col, "mean") for col in prob_cols}
        )
        rec_preds = rec_df[prob_cols].values.argmax(axis=1)

        rec_metrics.append({
            "fold": k,
            "Macro-F1": f1_score(rec_df["y_true"], rec_preds, average="macro"),
            "Balanced_Acc": balanced_accuracy_score(rec_df["y_true"], rec_preds),
            "MCC": matthews_corrcoef(rec_df["y_true"], rec_preds)
        })

    win_df = pd.DataFrame(win_metrics)
    rec_df_summary = pd.DataFrame(rec_metrics)

    print(f"\n==================== {model_name} (5-Fold CV) ====================")
    print("1. Window-Level (Mean ± Std across 5 folds):")
    for col in ["Macro-F1", "Balanced_Acc", "MCC"]:
        print(f"   {col:14s}: {win_df[col].mean():.4f} ± {win_df[col].std():.4f}")

    print("\n2. Recording-Level (Mean ± Std across 5 folds):")
    for col in ["Macro-F1", "Balanced_Acc", "MCC"]:
        print(f"   {col:14s}: {rec_df_summary[col].mean():.4f} ± {rec_df_summary[col].std():.4f}")

    # ج) النتيجة لكل داتاسيت على حدة (على مستوى التسجيل)
    full_val = pd.concat(all_val_rows, ignore_index=True)
    prob_cols = [f"prob_{i}" for i in range(len(classes))]
    rec_all = full_val.groupby("audio_path").agg(
        y_true=("target", "first"),
        dataset=("dataset", "first"),
        **{col: (col, "mean") for col in prob_cols}
    )
    rec_all["y_pred"] = rec_all[prob_cols].values.argmax(axis=1)

    ds_rows = []
    for ds_name, g in rec_all.groupby("dataset"):
        present_labels = np.unique(g["y_true"])  # يحسب فقط الفئات الموجودة فعلياً في هذه الداتاسيت
        ds_rows.append({
            "Dataset": ds_name,
            "Recordings": len(g),
            "Macro-F1 (Present Classes)": round(f1_score(g["y_true"], g["y_pred"], labels=present_labels, average="macro", zero_division=0), 4),
            "Balanced_Acc": round(balanced_accuracy_score(g["y_true"], g["y_pred"]), 4),
            "MCC": round(matthews_corrcoef(g["y_true"], g["y_pred"]), 4)
        })
    print("\n3. Per-Dataset Breakdown (Recording-Level):")
    display(pd.DataFrame(ds_rows))
    return win_df, rec_df_summary

# ==============================================================================

# ==============================================================================

# المودل الأول: MLP (الشبكة العصبية متعددة الطبقات - مع موازنة الأوزان)
win_mlp, rec_mlp = evaluate_model_cv(
    "MLP Neural Network (256, 128)",
    lambda: MLPClassifier(
        hidden_layer_sizes=(256, 128),
        activation="relu",
        alpha=1e-3,
        learning_rate_init=1e-3,
        max_iter=200,
        early_stopping=True,
        random_state=42
    ),
    use_sample_weight=True
)

# المودل الثاني: Cubic SVM (   من ورقة Özseven 2023)
win_cubic, rec_cubic = evaluate_model_cv(
    "Cubic SVM (Özseven 2023 - degree=3)",
    lambda: SVC(
        kernel="poly",
        degree=3,
        C=1.0,
        class_weight="balanced",
        probability=True,
        random_state=42
    ),
    use_sample_weight=False
)

# المودل الثالث: k-NN (Distance-weighted)
win_knn, rec_knn = evaluate_model_cv(
    "k-NN (k=7, distance-weighted)",
    lambda: KNeighborsClassifier(n_neighbors=7, weights="distance", n_jobs=-1),
    use_sample_weight=False
)

Google Drive is already mounted.
Total windows: 7450 | Number of features: 215
Classes: ['Burping', 'Discomfort', 'Hunger', 'Pain', 'Tired']

==================== MLP Neural Network (256, 128) (5-Fold CV) ====================
1. Window-Level (Mean ± Std across 5 folds):
   Macro-F1      : 0.6475 ± 0.0561
   Balanced_Acc  : 0.6363 ± 0.0575
   MCC           : 0.6240 ± 0.0724

2. Recording-Level (Mean ± Std across 5 folds):
   Macro-F1      : 0.7314 ± 0.0481
   Balanced_Acc  : 0.7197 ± 0.0435
   MCC           : 0.7009 ± 0.0598

3. Per-Dataset Breakdown (Recording-Level):


,Dataset,Recordings,Macro-F1 (Present Classes),Balanced_Acc,MCC
0,Baby Chillanto,441,0.6493,0.6367,0.2990
1,Donate-a-Cry,364,0.1806,0.1993,0.0518
2,InfantCry-DBL,809,0.7108,0.7296,0.6988
3,OSF,256,0.7595,0.7592,0.5196



==================== Cubic SVM (Özseven 2023 - degree=3) (5-Fold CV) ====================
1. Window-Level (Mean ± Std across 5 folds):
   Macro-F1      : 0.6427 ± 0.0461
   Balanced_Acc  : 0.6229 ± 0.0486
   MCC           : 0.6325 ± 0.0629

2. Recording-Level (Mean ± Std across 5 folds):
   Macro-F1      : 0.7177 ± 0.0378
   Balanced_Acc  : 0.7014 ± 0.0362
   MCC           : 0.6952 ± 0.0460

3. Per-Dataset Breakdown (Recording-Level):


,Dataset,Recordings,Macro-F1 (Present Classes),Balanced_Acc,MCC
0,Baby Chillanto,441,0.6729,0.6647,0.3452
1,Donate-a-Cry,364,0.1804,0.2000,0.0000
2,InfantCry-DBL,809,0.6658,0.6782,0.6528
3,OSF,256,0.7565,0.7410,0.5127



==================== k-NN (k=7, distance-weighted) (5-Fold CV) ====================
1. Window-Level (Mean ± Std across 5 folds):
   Macro-F1      : 0.5648 ± 0.0362
   Balanced_Acc  : 0.5512 ± 0.0413
   MCC           : 0.5340 ± 0.0418

2. Recording-Level (Mean ± Std across 5 folds):
   Macro-F1      : 0.6608 ± 0.0276
   Balanced_Acc  : 0.6488 ± 0.0334
   MCC           : 0.6189 ± 0.0309

3. Per-Dataset Breakdown (Recording-Level):


,Dataset,Recordings,Macro-F1 (Present Classes),Balanced_Acc,MCC
0,Baby Chillanto,441,0.6347,0.6267,0.2697
1,Donate-a-Cry,364,0.1787,0.1967,-0.0306
2,InfantCry-DBL,809,0.5903,0.6444,0.5528
3,OSF,256,0.7253,0.6952,0.4385
